<h1 align="center"><b>Laser Beam Scattering in Fog</b></h1>


## Numerical reproduction of Mie-based fog optical properties and Monte Carlo photon transport

This notebook reproduces and extends selected numerical results from Xu et al. (2023) on laser propagation in advection and radiation fog. It computes fog bulk optical properties from droplet size distributions using Mie theory, then uses Monte Carlo photon transport to study transmission, reflection, absorption, angular scattering intensity, and photon trajectory visualization.



## Table of Contents

1. [Introduction](#1.-Introduction)
   - [1.1 Motivation and Scope](#1.1-Motivation-and-Scope)
   - [1.2 Relation to the Reference Paper](#1.2-Relation-to-the-Reference-Paper)
   - [1.3 Reproduced and Extended Results](#1.3-Reproduced-and-Extended-Results)
2. [Computational Setup](#2.-Computational-Setup)
   - [2.1 Python Libraries and Plotting Configuration](#2.1-Python-Libraries-and-Plotting-Configuration)
   - [2.2 Output Directories and Reproducibility Settings](#2.2-Output-Directories-and-Reproducibility-Settings)
   - [2.3 Role of `miepython`](#2.3-Role-of-miepython)
3. [Fog Microphysics](#3.-Fog-Microphysics)
   - [3.1 Advection Fog and Radiation Fog](#3.1-Advection-Fog-and-Radiation-Fog)
   - [3.2 Fog Droplet Size Distribution](#3.2-Fog-Droplet-Size-Distribution)
   - [3.3 Visibility, Liquid Water Content, and Droplet Concentration](#3.3-Visibility,-Liquid-Water-Content,-and-Droplet-Concentration)
   - [3.4 Numerical Implementation of Fog Distribution Models](#3.4-Numerical-Implementation-of-Fog-Distribution-Models)
4. [Mie Scattering and Bulk Optical Properties](#4.-Mie-Scattering-and-Bulk-Optical-Properties)
   - [4.1 Single-Droplet Mie Efficiencies](#4.1-Single-Droplet-Mie-Efficiencies)
   - [4.2 Bulk Extinction, Scattering Albedo, and Asymmetry Factor](#4.2-Bulk-Extinction,-Scattering-Albedo,-and-Asymmetry-Factor)
   - [4.3 Numerical Integration over Droplet Radii](#4.3-Numerical-Integration-over-Droplet-Radii)
   - [4.4 Optical-Property Plots versus Visibility](#4.4-Optical-Property-Plots-versus-Visibility)
5. [Monte Carlo Photon Transport Model](#5.-Monte-Carlo-Photon-Transport-Model)
   - [5.1 Photon Free Path Sampling](#5.1-Photon-Free-Path-Sampling)
   - [5.2 Absorption Models: Analog and Implicit Weighting](#5.2-Absorption-Models:-Analog-and-Implicit-Weighting)
   - [5.3 Henyey-Greenstein Scattering Phase Function](#5.3-Henyey-Greenstein-Scattering-Phase-Function)
   - [5.4 Direction Update after Scattering](#5.4-Direction-Update-after-Scattering)
   - [5.5 Photon Fates: Transmitted, Reflected, Absorbed](#5.5-Photon-Fates:-Transmitted,-Reflected,-Absorbed)
6. [Photon Trajectory Visualization](#6.-Photon-Trajectory-Visualization)
   - [6.1 Two-Dimensional Photon Paths](#6.1-Two-Dimensional-Photon-Paths)
   - [6.2 Three-Dimensional Photon Paths](#6.2-Three-Dimensional-Photon-Paths)
   - [6.3 Animated Photon Transport](#6.3-Animated-Photon-Transport)
   - [6.4 Propagation-Depth Comparison](#6.4-Propagation-Depth-Comparison)
7. [Angular Scattering Intensity](#7.-Angular-Scattering-Intensity)
   - [7.1 Intensity Binning and Solid-Angle Normalization](#7.1-Intensity-Binning-and-Solid-Angle-Normalization)
   - [7.2 Albedo Sweep: Reproduction of Figure 5](#7.2-Albedo-Sweep:-Reproduction-of-Figure-5)
   - [7.3 Asymmetry-Factor Sweep: Reproduction of Figure 6](#7.3-Asymmetry-Factor-Sweep:-Reproduction-of-Figure-6)
   - [7.4 Backscattering by Fog Type and Wavelength: Reproduction of Figure 7](#7.4-Backscattering-by-Fog-Type-and-Wavelength:-Reproduction-of-Figure-7)
8. [Discussion](#8.-Discussion)
   - [8.1 Microphysics and Bulk Optical Properties ](#81-microphysics-and-bulk-optical-properties-figures-14)
   - [8.2 Spatial Trajectory & Diffusion Dynamics](#82-spatial-trajectory--diffusion-dynamics-2d3d-plots--animations)
   - [8.3 Angular Scattering Distributions & Parameter Sweeps](#83-angular-scattering-distributions--parameter-sweeps-figures-57)
   - [8.4 Engineering & System Architecture Implications](#84-engineering--system-architecture-implications-fso-vs-lidar)
9. [References](#9.-References)


## 1. Introduction

### **1.1 Motivation and Scope**
Laser beam propagation through atmospheric fog is critical for applications in free-space optical (FSO) communications, autonomous vehicle LiDAR, and remote sensing. Fog consists of suspended water droplets that scatter and absorb propagating photons, causing severe beam attenuation, angular dispersion, and multipath time broadening. Quantifying these effects requires modeling both the microphysical properties of fog droplets and the macroscopic radiative transfer of photons.

### **1.2 Relation to the Reference Paper**
This notebook serves as a rigorous numerical reproduction and modular refactor of the methodology presented in **Xu et al. (2023)** (*The Multiple Scattering of Laser Beam Propagation in Advection Fog and Radiation Fog*, International Journal of Optics). While the reference paper investigates multiple scattering effects using Mie theory and Monte Carlo simulations, this notebook refactors the workflows into a clean, reproducible, object-oriented Python implementation.

### **1.3 Reproduced and Extended Results**
Specifically, this notebook reproduces and extends:
1. **Bulk Optical Properties**: Visibility-dependent extinction coefficient ($\mu_e$), single scattering albedo ($\omega_0$), and asymmetry factor ($\langle g \rangle$) across five near- and mid-infrared laser wavelengths ($0.86, 0.91, 1.06, 1.315,$ and $10.6\,\mu\text{m}$).
2. **Angular Scattering Intensity Sweeps**: Forward and backward angular scattering distributions reproducing **Figure 5** (albedo sweep) and **Figure 6** (asymmetry factor sweep) from Xu et al. (2023).
3. **Backscattering Wavelength Sensitivity**: Backscatter intensity distribution versus scattering angle reproducing **Figure 7**, contrasting advection fog and radiation fog.
4. **Trajectory Visualizations**: 2D, 3D, and animated photon paths classifying photon fates into transmitted, reflected, and absorbed populations.


## 2. Computational Setup

### **2.1 Python Libraries and Plotting Configuration**
We employ standard scientific Python libraries: `numpy` for vectorized numerical arrays, `scipy.integrate.trapezoid` for integration over droplet size distributions, `scipy.signal.savgol_filter` for optional variance smoothing of Monte Carlo angular histograms, and `matplotlib` for 2D/3D visualization and animation.


In [ ]:
import json
from pathlib import Path
import hashlib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.animation import FuncAnimation
from scipy.signal import savgol_filter
from scipy.integrate import trapezoid
import miepython as mie
import datetime

# Configure standard Matplotlib styling
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "figure.titlesize": 16,
        "figure.titleweight": "bold",
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.labelsize": 12,
        "axes.labelweight": "bold",
        "legend.fontsize": 10,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "figure.dpi": 100,
        "axes.grid": True,
        "axes.grid.which": "both",
        "grid.color": "grey",
        "grid.linewidth": 0.5,
        "grid.alpha": 0.4,
    }
)


# timestamp for saving files
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")

# Uncomment for displaying animations in Jupyter Notebook
%matplotlib ipympl


### **2.2 Output Directories and Reproducibility Settings**
To ensure complete reproducibility and clean organization, all numerical matrices are cached as compressed `.npz` files alongside descriptive `.json` metadata files. Plots and animations are saved to dedicated subdirectories.


In [ ]:
OUTPUT_DIR = Path("OUTPUTS")
PLOT_DIR = OUTPUT_DIR / "PLOTS"
DATA_DIR = OUTPUT_DIR / "DATA"
ANIMATION_DIR = OUTPUT_DIR / "ANIMATIONS"

for folder in [OUTPUT_DIR, PLOT_DIR, DATA_DIR, ANIMATION_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Output directories established under: {OUTPUT_DIR.resolve()}")

### **2.3 Role of `miepython`**
We utilize the `miepython` library, a pure-Python implementation of Lorenz-Mie scattering theory for homogeneous spheres. Given the complex refractive index $m = n - ik$ of water at a specific wavelength $\lambda$ and the dimensionless size parameter $x = 2\pi r / \lambda$, `miepython.efficiencies(m, d, lam)` computes the single-droplet extinction efficiency $Q_e$, scattering efficiency $Q_s$, backscatter efficiency $Q_{back}$, and anisotropy factor $g$.


## 3. Fog Microphysics

### **3.1 Advection Fog and Radiation Fog**
Fog is classified by its meteorological formation mechanism:
- **Advection Fog**: Formed when warm, moist air moves horizontally over a cooler surface (e.g., coastal fog). It is characterized by higher wind speeds, larger liquid water content (LWC), and broader droplet size distributions containing larger droplets.
- **Radiation Fog**: Formed by nighttime radiative cooling of the ground under calm, clear skies. It features smaller droplets, lower LWC, and a narrower size distribution.

### **3.2 Fog Droplet Size Distribution**
The empirical droplet size distribution $n(r)$ represents the number of droplets per unit volume per unit radius interval. Following Xu et al. (2023), the distributions follow a modified gamma distribution parameterized by visibility $V$ (in km) and droplet radius $r$ (in $\mu\text{m}$):

$$n(r) = a r^2 e^{-br}$$

For **Advection Fog**:
$$n_{\mathrm{adv}}(r) = 1.059 \times 10^7 V^{1.15} r^2 \exp\left(-0.8359 V^{0.43} r\right)$$

For **Radiation Fog**:
$$n_{\mathrm{rad}}(r) = 3.104 \times 10^{10} V^{1.7} r^2 \exp\left(-4.122 V^{0.54} r\right)$$

Here, $n(r)$ has units of $\text{m}^{-3}\mu\text{m}^{-1}$.

### **3.3 Visibility, Liquid Water Content, and Droplet Concentration**
Atmospheric visibility $V$ is inversely proportional to extinction. As visibility drops from $10\,\text{km}$ (light haze) to $0.01\,\text{km}$ ($10\,\text{m}$, dense fog), droplet concentration and liquid water content rise drastically. Typical fog droplet radii span from $0.1\,\mu\text{m}$ up to $40\,\mu\text{m}$.


### **3.4 Numerical Implementation of Fog Distribution Models**
Below we define the refractive index table for water at laser wavelengths and implement the size distribution functions along with a visualization helper.


In [ ]:
# Refractive index table m = n - i*k. Imaginary part is negative representing absorption.
FOG_REF_INDEX = {
    0.86: complex(1.329, -2.93e-7),
    0.91: complex(1.323, -5.70e-7),
    1.06: complex(1.326, -2.89e-6),
    1.315: complex(1.316, -1.639e-5),
    10.6: complex(1.178, -0.071),
}


def n_adv(r_um, V_km):
    """
    Advection fog particle size distribution.

    Parameters:
        r_um (float or np.ndarray): Droplet radius in micrometers (um).
        visibility_km (float): Atmospheric visibility V in kilometers (km).

    Returns:
        np.ndarray: Droplet concentration n(r) in number / (m^3 * um).
    """
    return 1.059e7 * (V_km**1.15) * (r_um**2) * np.exp(-0.8359 * (V_km**0.43) * r_um)


def n_rad(r_um, V_km):
    """
    Radiation fog particle size distribution.

    Parameters:
        r_um (float or np.ndarray): Droplet radius in micrometers (um).
        visibility_km (float): Atmospheric visibility V in kilometers (km).

    Returns:
        np.ndarray: Droplet concentration n(r) in number / (m^3 * um).
    """
    return 3.104e10 * (V_km**1.7) * (r_um**2) * np.exp(-4.122 * (V_km**0.54) * r_um)


def plot_fog_distribution(visibilities=[0.1, 0.5, 2.0], save=False, filename=None):
    """Plots droplet size distribution for advection and radiation fogs across visibilities"""
    r = np.linspace(0.1, 25.0, 500)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Fog Droplet Size Distributions $n(r)$ across Visibilities")

    colors = plt.cm.plasma(np.linspace(0, 1, len(visibilities)))  # type:ignore
    for V, color in zip(visibilities, colors):
        n_rad_vals = n_rad(r, V)
        n_adv_vals = n_adv(r, V)

        peak_r_rad = r[np.argmax(n_rad_vals)]
        ax1.plot(r, n_rad_vals, color=color, lw=2, label=f"V = {V} km")
        ax1.axvline(x=peak_r_rad, color=color, ls="--", lw=0.8, alpha=0.7)

        peak_r_adv = r[np.argmax(n_adv_vals)]
        ax2.plot(r, n_adv_vals, color=color, lw=2, label=f"V = {V} km")
        ax2.axvline(x=peak_r_adv, color=color, ls="--", lw=0.8, alpha=0.7)

    ax1.set_title("Radiation Fog")
    ax2.set_title("Advection Fog")
    for ax in [ax1, ax2]:
        ax.set_xlabel("Droplet Radius $r$ ($\\mu$m)")
        ax.set_ylabel("Number Density $n(r)$ (m$^{-3}$ $\\mu$m$^{-1}$)")
        ax.legend(ncol=1 if len(visibilities) < 4 else 2)
        ax.set_ylim(0)
        ax.set_xlim(0, max(r))
    plt.tight_layout()
    if save:
        if filename is None:
            filename = f"fog_droplet_distributions_{timestamp}.png"
        save_path = PLOT_DIR / filename
        plt.savefig(save_path)
        print(f"Figure is saved to {save_path.resolve()}")
    plt.show()


visibilities = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]
plot_fog_distribution(visibilities=visibilities, save=True)


## 4. Mie Scattering and Bulk Optical Properties



### **4.1 Single-Droplet Mie Efficiencies**


For a spherical water droplet of radius $r$, Mie scattering theory yields dimensionless efficiencies $Q_e(r, \lambda, m)$ and $Q_s(r, \lambda, m)$ representing the geometric cross-section scaling for total extinction and scattering, respectively.

### **4.2 Bulk Extinction, Scattering Albedo, and Asymmetry Factor**

When a collimated laser beam propagates through a polydisperse fog layer consisting of millions of water droplets of varying sizes, macroscopic light transport is governed by three fundamental bulk optical properties. These macroscopic parameters are obtained by weighting individual single-droplet Mie scattering properties across the entire particle size distribution $n(r)$.


#### **Extinction Coefficient ($\mu_e$, $\mu_s$, and $\mu_a$)**
The **Extinction Coefficient** $\mu_e$ quantifies the fractional rate of attenuation of unscattered (ballistic) laser intensity per unit propagation distance $z$, following the classical **Beer-Lambert-Bouguer Law**:

$$I_{\text{ballistic}}(z) = I_0 \exp(-\mu_e z)$$

Extinction is the sum of two distinct physical mechanisms: **Scattering** ($\mu_s$, where photons change direction without energy loss) and **Absorption** ($\mu_a$, where photon energy is converted into thermal energy within the water droplet):

$$\mu_e = \mu_s + \mu_a$$

To compute the macroscopic coefficients across a polydisperse cloud, the physical cross-sectional area of each droplet bin ($\pi r^2$) is weighted by the dimensionless Mie efficiencies ($Q_e, Q_s$) and integrated over all droplet radii:

$$\mu_e = \pi \int_0^\infty r^2 Q_e(r, \lambda, m) n(r)\,dr$$

$$\mu_s = \pi \int_0^\infty r^2 Q_s(r, \lambda, m) n(r)\,dr$$

$$\mu_a = \mu_e - \mu_s = \pi \int_0^\infty r^2 Q_a(r, \lambda, m) n(r)\,dr$$

* **Physical Intuition:** The product $\mu_e z = \tau$ defines the **Optical Depth** ($\tau$) of the medium. At $\tau = 1$, the unscattered ballistic beam intensity drops to $1/e \approx 36.8\%$. For $\tau > 5\text{–}10$, the unscattered beam is virtually extinguished, and light reaches a receiver strictly through multiple diffuse scattering paths.


#### **Single-Scattering Albedo ($\omega_0$)**
The **Single-Scattering Albedo** $\omega_0$ represents the fraction of total optical attenuation that is caused by scattering versus thermal absorption:

$$\omega_0 = \frac{\mu_s}{\mu_e} = \frac{\mu_s}{\mu_s + \mu_a}$$

* **Physical Intuition:** $\omega_0$ represents the exact **survival probability** ($0 \le \omega_0 \le 1$) of a photon during a single collision event with a fog droplet:
  * **Conservative (Elastic) Scattering ($\omega_0 \to 1.0$):** Typical of near-infrared wavelengths ($0.86\text{ to }1.315\,\mu\text{m}$) where water is transparent ($\mu_a \approx 0$). Photons bounce off droplets repeatedly without losing energy. Because photons survive hundreds of collisions, they build up intense forward-diffuse halos and measurable backscattered signals.
  * **Absorbing Medium ($\omega_0 < 1.0$):** Typical of the far-infrared $10.6\,\mu\text{m}$ laser ($\omega_0 \approx 0.3\text{ to }0.6$). A significant fraction of photons are absorbed on each impact ($\mu_a > 0$). This internal absorption rapidly quenches higher-order multiple scattering and eliminates backscattered glare.


#### **Asymmetry Factor ($\langle g \rangle$)**
The **Asymmetry Factor** $\langle g \rangle$ measures the directional anisotropy of scattered light. It is defined as the ensemble-averaged cosine of the scattering deflection angle $\theta$ ($\langle g \rangle = \langle \cos \theta \rangle$), weighted by the scattering cross section across all droplet radii:

$$\langle g \rangle = \frac{\int_0^\infty r^2 Q_s(r, \lambda, m) g(r, \lambda, m) n(r)\,dr}{\int_0^\infty r^2 Q_s(r, \lambda, m) n(r)\,dr}$$

* **Physical Intuition & Scattering Regimes:**
  * **Strong Forward Scattering ($\langle g \rangle \to 1.0$):** When droplet sizes are much larger than the wavelength ($2\pi r \gg \lambda$, as in near-IR fog scattering where $\langle g \rangle \approx 0.75\text{–}0.88$), droplets act as microscopic lenses that diffract and refract light into a tight forward cone ($\theta < 15^\circ$). This allows optical momentum to carry deep into the fog layer before lateral diffusion takes over.
  * **Isotropic Scattering ($\langle g \rangle \approx 0.0$):** When droplet dimensions are smaller than or comparable to the wavelength ($2\pi r \lesssim \lambda$, such as $10.6\,\mu\text{m}$ probing microscopic radiation fog), scattering transitions toward the Rayleigh regime, scattering surviving photons equally into forward and backward hemispheres.
  * **Pure Backscattering ($\langle g \rangle = -1.0$):** Hypothetical complete reflection where all photons are turned $180^\circ$ backward upon collision.

#### **Unit Conventions & Scaling Factor**
Since droplet radius $r$ is specified in micrometers ($\mu\text{m} = 10^{-6}\,\text{m}$) and the size distribution $n(r)$ is expressed in $\text{m}^{-3}\mu\text{m}^{-1}$, evaluating the integral with $dr$ in $\mu\text{m}$ requires an exact scaling transformation:

$$\mu_e\ (\text{m}^{-1}) = 10^{-12} \times \pi \int r^2\ (\mu\text{m}^2) \cdot Q_e \cdot n(r)\ (\text{m}^{-3}\mu\text{m}^{-1})\,dr\ (\mu\text{m})$$

* Multiplying by $10^{-12}$ yields the extinction coefficient in standard **inverse meters** ($\text{m}^{-1}$).
* Multiplying $\mu_e\ (\text{m}^{-1})$ by $1000$ yields **inverse kilometers** ($\text{km}^{-1}$), the standard units used in atmospheric optics and visibility calculations. Both unit scales are computed and preserved within our `FogOpticalModel` class for complete engineering compatibility.

### **4.3 Numerical Integration over Droplet Radii**

Below we encapsulate the grid generation, Mie efficiency integration, saving, loading, and plotting into a clean object-oriented class `FogOpticalModel`.

In [ ]:
class FogOpticalModel:
    """Compute, cache, and plot bulk fog optical properties from size distributions."""

    def __init__(
        self, r_min=0.1, r_max=40.0, num_r=1000, v_min=0.01, v_max=10.0, num_v=50
    ):
        self.r_um = np.logspace(np.log10(r_min), np.log10(r_max), num_r)
        self.visibility_km = np.logspace(np.log10(v_min), np.log10(v_max), num_v)
        self.wavelengths_um = [0.86, 0.91, 1.06, 1.315, 10.6]
        self.colors = ["red", "mediumseagreen", "royalblue", "cyan", "black"]
        self.ref_indices = FOG_REF_INDEX
        self.fog_models = {"Radiation Fog": n_rad, "Advection Fog": n_adv}
        self.results = {}

    def compute_single_property(self, n_func, visibility_km, wavelength_um):
        """Compute bulk mu_e (km^-1), omega0, and asymmetry factor g."""
        m = self.ref_indices[wavelength_um]
        Qe_arr = np.zeros(len(self.r_um))
        Qs_arr = np.zeros(len(self.r_um))
        g_arr = np.zeros(len(self.r_um))

        for i, r in enumerate(self.r_um):
            Qe, Qs, _, g_val = mie.efficiencies(m, 2 * r, wavelength_um)
            Qe_arr[i] = Qe
            Qs_arr[i] = Qs
            g_arr[i] = g_val

        n_arr = n_func(self.r_um, visibility_km)
        weight = (self.r_um**2) * n_arr

        # Integrate: factor 1e-12 converts um^2 * m^-3 um^-1 * um to m^-1
        mu_e_m1 = np.pi * trapezoid(weight * Qe_arr, self.r_um) * 1e-12
        mu_s_m1 = np.pi * trapezoid(weight * Qs_arr, self.r_um) * 1e-12
        mu_e_km1 = mu_e_m1 * 1000.0

        omega0 = mu_s_m1 / mu_e_m1 if mu_e_m1 > 0 else 0.0
        g_avg = (
            trapezoid(weight * Qs_arr * g_arr, self.r_um)
            / trapezoid(weight * Qs_arr, self.r_um)
            if mu_s_m1 > 0
            else 0.0
        )

        return mu_e_km1, omega0, g_avg

    def compute_all_properties(self):
        """Execute grid sweeps across all fog models, wavelengths, and visibilities."""
        self.results = {}
        for fog_name, n_func in self.fog_models.items():
            self.results[fog_name] = {}
            for wl in self.wavelengths_um:
                mu_e_list, w0_list, g_list = [], [], []
                for V in self.visibility_km:
                    mu_e, w0, g_val = self.compute_single_property(n_func, V, wl)
                    mu_e_list.append(mu_e)
                    w0_list.append(w0)
                    g_list.append(g_val)
                self.results[fog_name][str(wl)] = {
                    "mu_e": np.array(mu_e_list),
                    "omega0": np.array(w0_list),
                    "g": np.array(g_list),
                }
        return self.results

    def save_properties(
        self,
        data_path=DATA_DIR / "fog_optical_properties.npz",
        metadata_path=DATA_DIR / "fog_optical_properties_metadata.json",
    ):
        """Save computed data arrays to .npz and metadata to .json."""
        if not self.results:
            self.compute_all_properties()
        npz_dict = {"visibility_km": self.visibility_km}
        for fog_name, wl_dict in self.results.items():
            prefix = fog_name.lower().replace(" ", "_")
            for wl_str, props in wl_dict.items():
                npz_dict[f"{prefix}_{wl_str}_mu_e"] = props["mu_e"]
                npz_dict[f"{prefix}_{wl_str}_omega0"] = props["omega0"]
                npz_dict[f"{prefix}_{wl_str}_g"] = props["g"]
        np.savez_compressed(data_path, **npz_dict)

        metadata = {
            "wavelengths_um": self.wavelengths_um,
            "fog_models": list(self.fog_models.keys()),
            "units": {"mu_e": "km^-1", "visibility": "km", "wavelength": "um"},
            "description": "Mie bulk optical properties for advection and radiation fog.",
        }
        with open(metadata_path, "w") as f:
            json.dump(metadata, f, indent=4)
        print(f"Saved optical properties dataset to {data_path}")

    def load_properties(
        self,
        data_path=DATA_DIR / "fog_optical_properties.npz",
        metadata_path=DATA_DIR / "fog_optical_properties_metadata.json",
    ):
        """Load pre-computed optical property grids from disk."""
        with open(metadata_path, "r") as f:
            meta = json.load(f)
        data = np.load(data_path)
        self.visibility_km = data["visibility_km"]
        self.results = {}
        for fog_name in meta["fog_models"]:
            self.results[fog_name] = {}
            prefix = fog_name.lower().replace(" ", "_")
            for wl in meta["wavelengths_um"]:
                wl_str = str(wl)
                self.results[fog_name][wl_str] = {
                    "mu_e": data[f"{prefix}_{wl_str}_mu_e"],
                    "omega0": data[f"{prefix}_{wl_str}_omega0"],
                    "g": data[f"{prefix}_{wl_str}_g"],
                }
        print(f"Loaded optical properties dataset from {data_path}")

    def run_or_load_properties(
        self,
        data_path=DATA_DIR / "fog_optical_properties.npz",
        metadata_path=DATA_DIR / "fog_optical_properties_metadata.json",
        force_recompute=False,
    ):
        """Load pre-computed optical properties if .npz and .json files exist on disk; otherwise compute and save them."""
        if data_path.exists() and metadata_path.exists() and not force_recompute:
            self.load_properties(data_path, metadata_path)
        else:
            self.compute_all_properties()
            self.save_properties(data_path, metadata_path)

    def plot_property(self, property_name, save=False, save_filename=None):
        """Plot mu_e, omega0, or g against atmospheric visibility."""
        prop_map = {
            "mu_e": {
                "key": "mu_e",
                "ylabel": "Avg. Extinction $\\langle \\mu_e$ \\rangle (km$^{-1}$)",
                "title": "Average Extinction Coefficient vs Visibility",
                "loglog": True,
            },
            "w": {
                "key": "omega0",
                "ylabel": "Single Scattering Albedo $\\omega_0$",
                "title": "Single Scattering Albedo vs Visibility",
                "loglog": False,
            },
            "g": {
                "key": "g",
                "ylabel": "Asymmetry Factor $\\langle g \\rangle$",
                "title": "Average Asymmetry Factor vs Visibility",
                "loglog": False,
            },
        }
        info = prop_map[property_name]
        key = info["key"]

        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        fig.suptitle(info["title"], y=0.98)

        for ax, (fog_name, wl_dict) in zip(axes, self.results.items()):
            for wl, color in zip(self.wavelengths_um, self.colors):
                vals = wl_dict[str(wl)][key]
                if info["loglog"]:
                    ax.loglog(
                        self.visibility_km,
                        vals,
                        color=color,
                        lw=1.8,
                        marker="s",
                        ms=3,
                        label=f"{wl} $\\mu$m",
                    )
                else:
                    ax.semilogx(
                        self.visibility_km,
                        vals,
                        color=color,
                        lw=1.8,
                        marker="s",
                        ms=3,
                        label=f"{wl} $\\mu$m",
                    )
            ax.set_title(fog_name)
            ax.set_xlim(0.01, 10)
            ax.set_xticks(ticks=[0.01, 0.1, 1, 10], labels=["0.01", "0.1", "1", "10"])
            ax.set_xlabel("Visibility $V$ (km)")
            ax.set_ylabel(info["ylabel"])

        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(
            handles,
            labels,
            loc="lower center",
            bbox_to_anchor=(0.5, -0.08),
            ncol=len(self.wavelengths_um),
        )

        if save and save_filename:
            if save_filename.endswith((".png", ".jpg", ".jpeg")):
                ext = save_filename.split(".")[-1]
                save_filename = save_filename.split(".")[0] + f"_{timestamp}.{ext}"
            else:
                save_filename += f"_{timestamp}.png"
            save_path = PLOT_DIR / save_filename
            plt.savefig(save_path, dpi=300, bbox_inches="tight")
            print(f"Figure saved to {save_path.resolve()}")
        plt.show()


fog_model = FogOpticalModel()
fog_model.run_or_load_properties()

### **4.4 Optical-Property Plots versus Visibility**


Below we plot single scattering albedo $\omega_0$, bulk extinction $\mu_e$, and asymmetry factor $\langle g \rangle$ across the full visibility range ($0.01\text{ to }10\,\text{km}$). Notice that at $10.6\,\mu\text{m}$ (mid-infrared), water absorbs strongly, leading to a noticeable drop in single-scattering albedo compared to near-infrared wavelengths.


In [ ]:
fog_model.plot_property("w", save=True, save_filename="fog_albedo_vs_visibility.png")


In [ ]:
fog_model.plot_property(
    "mu_e", save=True, save_filename="fog_extinction_vs_visibility.png"
)


In [ ]:
fog_model.plot_property("g", save=True, save_filename="fog_asymmetry_vs_visibility.png")


## 5. Monte Carlo Photon Transport Model (`PhotonTransportMC`)

We model the radiative transfer through the fog layer as a 3D Markov chain of independent stochastic photon-packet trajectories encapsulated in the `PhotonTransportMC` class. Each packet represents an ensemble of photons tracking position $(x, y, z)$, direction cosines $(\mu_x, \mu_y, \mu_z)$, statistical weight $w$, and interaction counts (`num_scatters`).


### **5.1 Path Initialization and Boundary Conditions (`__init__`)**
When `PhotonTransportMC.run()` is invoked, each photon packet is initialized at the entrance plane of the plane-parallel fog medium ($0 \le z \le z_{\max}$):
* **Spatial Initialization:** Photons enter at the origin of the front boundary face:
  $$(x_0, y_0, z_0) = (0.0, 0.0, 0.0)$$
* **Directional Initialization:** The incident laser beam is assumed perfectly collimated and perpendicular to the entrance boundary along the optical $Z$-axis:
  $$\vec{\mu}_0 = (\mu_x, \mu_y, \mu_z) = (0.0, 0.0, 1.0)$$
* **Weight Initialization:** Each packet begins with unit statistical weight $w = 1.0$ (`weight = 1.0`) and active tracking status `alive = True`.


### **5.2 Free Path Sampling (`run` loop)**
Before every scattering or absorption interaction, the photon travels an unscattered geometric step length $L$ sampled from the Beer-Lambert exponential attenuation distribution using a uniform random number $\xi \in (0, 1]$:

$$L = -\frac{\ln \xi}{\mu_e}$$

The candidate position for the next collision event is computed linearly along the current direction vector:

$$x_{\text{next}} = x + L \mu_x, \quad y_{\text{next}} = y + L \mu_y, \quad z_{\text{next}} = z + L \mu_z$$

If $z_{\text{next}}$ remains inside the medium ($0 < z_{\text{next}} < z_{\max}$), the photon updates its spatial coordinates (`x, y, z = next_x, next_y, next_z`), increments `num_scatters += 1`, and undergoes absorption and scattering.


### **5.3 Absorption Models: Analog vs. Implicit Weighting (`analog_absorption`)**
To accommodate both finite-path trajectory visualization and low-variance angular scattering calculations, `PhotonTransportMC` supports two distinct absorption modes governed by `self.analog_absorption`:

#### **Analog Absorption (`analog_absorption = True`)**
* At each internal collision step, a uniform random variable $\xi_{\text{abs}} \in [0, 1)$ is compared against the single-scattering albedo $\omega_0$ (`self.albedo`).
* If $\xi_{\text{abs}} > \omega_0$, the photon is **absorbed outright (`A_weight += weight`)**, its status is set to `alive = False`, and the trajectory terminates immediately.
* **Use Case:** This binary survival scheme ($w \in \{0, 1\}$) is computationally fast and produces discrete, physical 2D and 3D trajectory paths suitable for spatial rendering (`plot_2d_trajectories`, `visualize_mc_scattering_3d`).

#### **Implicit Weighting / Continuous Absorption (`analog_absorption = False`)**
* The photon packet is never killed outright by absorption. Instead, at every step, the exact fraction of absorbed energy is deposited locally into the medium (`A_weight += weight * (1.0 - self.albedo)`).
* The surviving packet weight is attenuated deterministically:
  $$w \leftarrow w \cdot \omega_0$$
* **Russian Roulette Termination:** Because tracking statistically negligible weights ($w \to 0$) indefinitely wastes computational time without altering macroscopic results, **Russian Roulette** is triggered when $w < 10^{-4}$ (`1e-4`):
  * A random variable $\xi_{\text{rr}} \in [0, 1)$ is sampled against a $10\%$ survival threshold (`0.1`).
  * If $\xi_{\text{rr}} < 0.1$, the photon survives and its weight is boosted tenfold ($w \leftarrow 10 w$) to strictly conserve energy expectation.
  * If $\xi_{\text{rr}} \ge 0.1$, the photon is terminated (`alive = False`).
* **Use Case:** Implicit weighting drastically reduces variance and is mandatory for computing smooth angular scattering intensity distributions (`return_weights = True`).


### **5.4 Henyey-Greenstein Scattering Phase Function (`sample_hg_angle`)**
If the photon survives absorption, its direction is redirected by scattering. Because exact Mie scattering phase functions are strongly forward-peaked, `PhotonTransportMC.sample_hg_angle()` models angular deflection using the analytical **Henyey-Greenstein (HG) phase function**:

$$p(\theta) = \frac{1 - g^2}{4\pi \left(1 + g^2 - 2g \cos \theta\right)^{3/2}}$$

Given a uniform random variable $\xi \in (0, 1)$, the scattering deflection cosine $\cos \theta$ is sampled exactly via cumulative distribution inversion:

$$\cos \theta = \begin{cases}
\frac{1}{2g} \left[ 1 + g^2 - \left( \frac{1 - g^2}{1 - g + 2g\xi} \right)^2 \right], & \text{if } g \ne 0 \\[10pt]
2\xi - 1, & \text{if } g = 0 \text{ (isotropic)}
\end{cases}$$


### **5.5 Direction Update after Scattering (`update_direction`)**
Given the deflection cosine $\cos \theta$, sine $\sin \theta = \sqrt{1 - \cos^2 \theta}$, and a uniformly sampled azimuthal angle around the previous trajectory axis $\phi = 2\pi \xi_{\phi} \in [0, 2\pi)$, `PhotonTransportMC.update_direction()` transforms the local deflection angles back into global laboratory direction cosines $(\mu_x', \mu_y', \mu_z')$ using 3D spherical rotation formulas:

$$
\begin{aligned}
\mu_x' &= \frac{\sin \theta (\mu_x \mu_z \cos \phi - \mu_y \sin \phi)}{\sqrt{1 - \mu_z^2}} + \mu_x \cos \theta \\

\mu_y' &= \frac{\sin \theta (\mu_y \mu_z \cos \phi + \mu_x \sin \phi)}{\sqrt{1 - \mu_z^2}} + \mu_y \cos \theta \\

\mu_z' &= -\sin \theta \cos \phi \sqrt{1 - \mu_z^2} + \mu_z \cos \theta
\end{aligned}
$$

#### Coordinate Singularity Protection (`abs(mu_z) > 0.99999`)
When the photon is traveling vertically along the $Z$-axis (such as at initial entry where $\mu_z = 1.0$), the denominator $\sqrt{1 - \mu_z^2} \to 0$. `update_direction()` explicitly handles this mathematical singularity, simplifying direction updates to:

$$\mu_x' = \sin \theta \cos \phi, \quad \mu_y' = \sin \theta \sin \phi, \quad \mu_z' = \text{sign}(\mu_z) \cos \theta$$


### **5.6 Photon Fates and Fate Determination (`run` & `classify_trajectory`)**

Every simulated photon trajectory terminates in one of three mutually exclusive macroscopic fates. During simulation execution (`run()`), boundary checks determine immediate exit states, while the static method `PhotonTransportMC.classify_trajectory(path_z, z_max)` allows post-processing categorization of any historical trajectory:

| Fate Classification | Boundary Condition in `run()` | Classification Rule in `classify_trajectory()` | Physical Interpretation & Action Taken |
| :--- | :--- | :--- | :--- |
| **Transmitted ($T$)** | `next_z >= self.z_max` | `last_z >= z_max * 0.999` | Photon successfully penetrates through the entire fog layer ($z \ge z_{\max}$). Weight is added to `T_weight += weight`. If the photon underwent multiple scattering (`num_scatters > 0`), its exit angle $\theta_f = \arccos(\mu_z)$ and weight are recorded in `forward_angles` for forward intensity analysis. |
| **Reflected ($R$)** | `next_z <= 0.0` | `last_z <= z_max * 0.001` | Photon backscatters out of the entrance boundary ($z \le 0$). Weight is added to `R_weight += weight`. Its backward escape angle $\theta_b = \arccos(-\mu_z)$ and weight are recorded in `backward_angles` for backscatter intensity analysis. |
| **Absorbed ($A$)** | `alive = False` while `0 < z < z_max` | `0.001 * z_max < last_z < 0.999 * z_max` | Photon energy is entirely dissipated inside the fog volume either via direct analog absorption or Russian Roulette termination. Accumulated into `A_weight += weight`. |

By strict conservation of energy across the `num_photons` ensemble, the macroscopic fractions satisfy exact energy balance:

$$T + R + A = 1.0$$

In [ ]:
class PhotonTransportMC:
    """Monte Carlo simulation of photon transport through plane-parallel fog."""

    def __init__(
        self,
        mu_e_km1,
        albedo,
        g,
        z_max_km,
        num_photons=1000,
        seed=42,
        analog_absorption=True,
    ):
        self.mu_e = mu_e_km1
        self.albedo = albedo
        self.g = g
        self.z_max = z_max_km
        self.num_photons = num_photons
        self.seed = seed
        self.analog_absorption = analog_absorption

    def sample_hg_angle(self):
        """Sample scattering cosine cos(theta) from Henyey-Greenstein function."""
        xi = np.random.rand()
        if self.g == 0:
            return 2.0 * xi - 1.0
        term = (1.0 - self.g**2) / (1.0 - self.g + 2.0 * self.g * xi)
        return (1.0 / (2.0 * self.g)) * (1.0 + self.g**2 - term**2)

    @staticmethod
    def update_direction(mu_x, mu_y, mu_z, cos_theta):
        """Update direction cosines after scattering event."""
        sin_theta = np.sqrt(max(0.0, 1.0 - cos_theta**2))
        phi = 2.0 * np.pi * np.random.rand()
        sin_phi, cos_phi = np.sin(phi), np.cos(phi)

        if abs(mu_z) > 0.99999:
            mu_x_new = sin_theta * cos_phi
            mu_y_new = sin_theta * sin_phi
            mu_z_new = np.sign(mu_z) * cos_theta
        else:
            denom = np.sqrt(1.0 - mu_z**2)
            mu_x_new = (
                sin_theta * (mu_x * mu_z * cos_phi - mu_y * sin_phi) / denom
                + mu_x * cos_theta
            )
            mu_y_new = (
                sin_theta * (mu_y * mu_z * cos_phi + mu_x * sin_phi) / denom
                + mu_y * cos_theta
            )
            mu_z_new = -sin_theta * cos_phi * denom + mu_z * cos_theta

        return mu_x_new, mu_y_new, mu_z_new

    def run(self, return_weights=False, stop_at_boundary=False):
        """Execute Monte Carlo simulation and return trajectories or angular arrays."""
        if self.seed is not None:
            np.random.seed(self.seed)

        forward_angles, forward_weights = [], []
        backward_angles, backward_weights = [], []
        trajectories = []
        T_weight, R_weight, A_weight = 0.0, 0.0, 0.0

        for _ in range(self.num_photons):
            x, y, z = 0.0, 0.0, 0.0
            mu_x, mu_y, mu_z = 0.0, 0.0, 1.0
            weight = 1.0
            path_x, path_y, path_z = [x], [y], [z]
            alive = True
            num_scatters = 0

            while alive:
                xi = np.random.rand()
                L = -np.log(max(1e-12, xi)) / self.mu_e
                next_x = x + L * mu_x
                next_y = y + L * mu_y
                next_z = z + L * mu_z

                # Transmission boundary check
                if next_z >= self.z_max:
                    if stop_at_boundary:
                        dz = self.z_max - z
                        frac = dz / max(1e-12, L * mu_z)
                        path_x.append(x + L * mu_x * frac)
                        path_y.append(y + L * mu_y * frac)
                        path_z.append(self.z_max)
                    else:
                        path_x.append(next_x)
                        path_y.append(next_y)
                        path_z.append(next_z)
                    alive = False
                    T_weight += weight
                    if num_scatters > 0:
                        forward_angles.append(np.arccos(np.clip(mu_z, -1.0, 1.0)))
                        forward_weights.append(weight)
                    break

                # Reflection boundary check
                elif next_z <= 0.0:
                    if stop_at_boundary:
                        dz = 0.0 - z
                        frac = dz / min(-1e-12, L * mu_z)
                        path_x.append(x + L * mu_x * frac)
                        path_y.append(y + L * mu_y * frac)
                        path_z.append(0.0)
                    else:
                        path_x.append(next_x)
                        path_y.append(next_y)
                        path_z.append(next_z)
                    alive = False
                    R_weight += weight
                    backward_angles.append(np.arccos(np.clip(-mu_z, -1.0, 1.0)))
                    backward_weights.append(weight)
                    break

                # Internal movement
                x, y, z = next_x, next_y, next_z
                path_x.append(x)
                path_y.append(y)
                path_z.append(z)
                num_scatters += 1

                # Absorption
                if self.analog_absorption:
                    if np.random.rand() > self.albedo:
                        A_weight += weight
                        alive = False
                        break
                else:
                    absorbed_frac = weight * (1.0 - self.albedo)
                    A_weight += absorbed_frac
                    weight *= self.albedo
                    if weight < 1e-4:
                        if np.random.rand() < 0.1:
                            weight *= 10.0
                        else:
                            alive = False
                            break

                # Scatter
                cos_theta = self.sample_hg_angle()
                mu_x, mu_y, mu_z = self.update_direction(mu_x, mu_y, mu_z, cos_theta)

            trajectories.append((path_x, path_y, path_z))

        T = T_weight / self.num_photons
        R = R_weight / self.num_photons
        A = A_weight / self.num_photons

        if return_weights:
            return (
                np.array(forward_angles),
                np.array(forward_weights),
                np.array(backward_angles),
                np.array(backward_weights),
            )
        return trajectories, T, R, A

    @staticmethod
    def classify_trajectory(path_z_km, z_max_km):
        """Classify fate as transmitted, reflected, or absorbed."""
        last_z = path_z_km[-1]
        if last_z >= z_max_km * 0.999:
            return "transmitted"
        elif last_z <= z_max_km * 0.001:
            return "reflected"
        return "absorbed"

    @staticmethod
    def calculate_intensity(angles_rad, weights, total_photons, num_bins=30):
        """Convert escape angles and weights to normalized solid-angle scattering intensity."""
        bin_edges = np.linspace(0, np.pi / 2, num_bins + 1)
        bin_centers_rad = (bin_edges[:-1] + bin_edges[1:]) / 2.0
        bin_centers_deg = np.rad2deg(bin_centers_rad)
        d_theta = bin_edges[1] - bin_edges[0]
        intensity = np.zeros(num_bins)

        bin_indices = np.digitize(angles_rad, bin_edges) - 1
        for i in range(num_bins):
            mask = bin_indices == i
            weight_sum = np.sum(weights[mask])
            solid_angle = 2.0 * np.pi * np.sin(bin_centers_rad[i]) * d_theta
            intensity[i] = weight_sum / max(1e-12, total_photons * solid_angle)

        return bin_centers_deg, intensity


## 6. Photon Trajectory Visualization

### **6.1 Two-Dimensional Photon Paths**

Visualizing projection paths along longitudinal depth ($Z$) and transverse spread ($X$) highlights how multiple scattering disperses a collimated laser beam.

In [ ]:
def plot_2d_trajectories(mc_model, save=False, save_filename=None):
    trajectories, T, R, A = mc_model.run(return_weights=False, stop_at_boundary=False)
    fig, ax = plt.subplots(figsize=(12, 6))
    for path_x, path_y, path_z in trajectories:
        ax.plot(path_z, path_x, alpha=0.6, lw=1)

    ax.axvline(0, color="black", linestyle="--", label="Entrance Boundary (Z=0)")
    ax.axvline(
        mc_model.z_max,
        color="red",
        linestyle="--",
        label=f"Exit Boundary (Z={mc_model.z_max} km)",
    )
    ax.fill_betweenx(
        [ax.get_ylim()[0], ax.get_ylim()[1]],
        0,
        mc_model.z_max,
        color="lightgray",
        alpha=0.6,
        label="Fog Medium",
    )

    ax.set_xlabel("Propagation Depth Z (km)")
    ax.set_ylabel("Transverse Spread X (km)")
    title = (
        f"Monte Carlo 2D Photon Trajectories (N={mc_model.num_photons})\n"
        f"$\\mu_e$={mc_model.mu_e} km$^{{-1}}$, $g$={mc_model.g}, $\\omega_0$={mc_model.albedo}\n"
        f"Transmitted={T * 100:.1f}% | Backscattered={R * 100:.1f}% | Absorbed={A * 100:.1f}%"
    )
    ax.set_title(title)
    ax.legend(loc="upper right")
    if save and save_filename:
        save_path = PLOT_DIR / save_filename
        plt.savefig(save_path, dpi=300)
        print(f"Figure saved to {save_path}")
    plt.show()


mc_2d = PhotonTransportMC(
    mu_e_km1=20.0, albedo=0.9, g=0.85, z_max_km=0.1, num_photons=100
)
plot_2d_trajectories(
    mc_2d,
    save=True,
    save_filename="mc_2d_trajectories.png",
)


### **6.2 Three-Dimensional Photon Paths**

Rendering static 3D paths with distinct color-coding for final photon fates clarifies spatial divergence and boundary escape dynamics. The 3D view can provide a better insight into the angular scattering distribution and the effect of multiple scattering events on photon trajectories.


In [ ]:
def visualize_mc_scattering_3d(
    mc_model,
    fps=15,
    animate=False,
    save_animation=False,
    save_figure=False,
    filename=None,
):
    print(
        f"Running 3D Monte Carlo Simulation for {mc_model.num_photons} photons (z_max={mc_model.z_max * 1000.0}m)..."
    )

    # Running the Monte Carlo simulation with analog absorption enabled
    mc_model.analog_absorption = True
    paths, T, R, A = mc_model.run(return_weights=False, stop_at_boundary=True)

    fate_colors = {
        "transmitted": "#4FC3F7",
        "reflected": "#EF5350",
        "absorbed": "#FFD54F",
    }

    fig = plt.figure(figsize=(12, 10))
    fig.patch.set_facecolor("#010408")
    fig.subplots_adjust(top=0.88, bottom=0.0, left=0.01, right=0.98)

    fig_title = (
        f"3D Monte Carlo Photon Transport\n"
        f"N={mc_model.num_photons}  ·  $\\mu_e$={mc_model.mu_e}/km  ·  g={mc_model.g}  ·  "
        f"$\\omega$={mc_model.albedo}  ·  Z = {mc_model.z_max * 1000.0}m"
    )
    fig.suptitle(
        fig_title,
        color="white",
    )

    ax = fig.add_subplot(111, projection="3d")
    ax.set_facecolor("#010408")

    max_spread = 0.0
    n_scatter_list = []

    for path_x, path_y, path_z in paths:
        p_x = np.array(path_x) * 1000.0
        p_y = np.array(path_y) * 1000.0
        current_max = max(np.max(np.abs(p_x)), np.max(np.abs(p_y)))
        max_spread = max(max_spread, current_max)
        n_scatter_list.append(len(path_x) - 1)

    max_spread = max(max_spread, 10.0)

    padded_spread = max_spread * 1.25

    max_n = max(n_scatter_list) if n_scatter_list else 1

    for (path_x, path_y, path_z), n_scatter in zip(paths, n_scatter_list):
        p_x = np.array(path_x) * 1000.0
        p_y = np.array(path_y) * 1000.0
        p_z = np.array(path_z) * 1000.0

        # Classify fate using the static method from PhotonTransportMC
        fate = PhotonTransportMC.classify_trajectory(path_z, mc_model.z_max)
        base_color = fate_colors[fate]

        alpha = max(0.15, 0.8 - 0.6 * (n_scatter / max_n))
        lw = max(0.4, 1.2 - 0.8 * (n_scatter / max_n))

        ax.plot(
            p_z,
            p_x,
            p_y,
            color=base_color,
            alpha=alpha if fate != "absorbed" else 0.85,
            linewidth=lw if fate != "absorbed" else 0.7,
            zorder=2 if fate != "absorbed" else 3,
        )

        # Mark the points where they cross the planes (or get absorbed)
        if fate == "absorbed":
            ax.scatter(
                [p_z[-1]],
                [p_x[-1]],
                [p_y[-1]],
                color=base_color,
                s=15,
                zorder=6,
                depthshade=False,
                edgecolors="white",
                linewidths=0.4,
            )
        elif fate == "transmitted" or fate == "reflected":
            ax.scatter(
                [p_z[-1]],
                [p_x[-1]],
                [p_y[-1]],
                color=base_color,
                s=15,
                marker="s",
                zorder=6,
                depthshade=False,
                edgecolors="white",
                linewidths=0.4,
            )

    # Grid planes (entrance & exit) with padded_spread
    grid_pts = np.linspace(-padded_spread, padded_spread, 10)
    yy, xx = np.meshgrid(grid_pts, grid_pts)

    zz0 = np.zeros_like(xx)
    ax.plot_surface(zz0, xx, yy, color="#00E676", alpha=0.1, zorder=1)  # type: ignore

    for i in range(len(grid_pts)):
        ax.plot(
            [0, 0],
            [grid_pts[i], grid_pts[i]],
            [grid_pts[0], grid_pts[-1]],
            color="#00E676",
            alpha=0.25,
            linewidth=0.5,
        )
        ax.plot(
            [0, 0],
            [grid_pts[0], grid_pts[-1]],
            [grid_pts[i], grid_pts[i]],
            color="#00E676",
            alpha=0.25,
            linewidth=0.5,
        )

    zz_max = np.full_like(xx, mc_model.z_max * 1000.0)
    ax.plot_surface(zz_max, xx, yy, color="#FFB300", alpha=0.1, zorder=1)  # type: ignore

    for i in range(len(grid_pts)):
        ax.plot(
            [mc_model.z_max * 1000.0, mc_model.z_max * 1000.0],
            [grid_pts[i], grid_pts[i]],
            [grid_pts[0], grid_pts[-1]],
            color="#FFB300",
            alpha=0.25,
            linewidth=0.5,
        )
        ax.plot(
            [mc_model.z_max * 1000.0, mc_model.z_max * 1000.0],
            [grid_pts[0], grid_pts[-1]],
            [grid_pts[i], grid_pts[i]],
            color="#FFB300",
            alpha=0.25,
            linewidth=0.5,
        )

    ax.set_xlim(-0.05 * mc_model.z_max * 1000.0, mc_model.z_max * 1000.0 * 1.05)
    ax.set_ylim(-padded_spread, padded_spread)
    ax.set_zlim(-padded_spread, padded_spread)  # type:ignore
    ax.set_axis_off()

    for pane in [ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane]:  # type:ignore
        pane.fill = False
        pane.set_edgecolor("#1a1f2e")

    marker_ax = fig.add_axes((0.80, 0.8, 0.14, 0.14), facecolor="none")
    marker_ax.set_xlim(0.0, 1.0)
    marker_ax.set_ylim(0.0, 1.0)
    marker_ax.axis("off")

    marker_ax.annotate(
        "",
        xy=(0.9, 0.5),
        xytext=(0.25, 0.5),
        arrowprops=dict(arrowstyle="->", lw=2.0, color="#4FC3F7"),
    )
    marker_ax.text(
        0.93, 0.5, "Z", color="#4FC3F7", fontsize=9, fontweight="bold", va="center"
    )

    marker_ax.annotate(
        "",
        xy=(0.25, 0.9),
        xytext=(0.25, 0.5),
        arrowprops=dict(arrowstyle="->", lw=2.0, color="#EF5350"),
    )
    marker_ax.text(
        0.23, 0.93, "X", color="#EF5350", fontsize=9, fontweight="bold", ha="right"
    )

    marker_ax.annotate(
        "",
        xy=(0.58, 0.2),
        xytext=(0.25, 0.5),
        arrowprops=dict(arrowstyle="->", lw=2.0, color="#00E676"),
    )
    marker_ax.text(
        0.61, 0.18, "Y", color="#00E676", fontsize=9, fontweight="bold", va="top"
    )

    legend_elements = [
        Line2D(
            [0],
            [0],
            color=fate_colors["transmitted"],
            lw=1.5,
            label=f"Transmitted ({T * 100:.1f}%)",
        ),
        Line2D(
            [0],
            [0],
            color=fate_colors["reflected"],
            lw=1.5,
            label=f"Reflected ({R * 100:.1f}%)",
        ),
        Line2D(
            [0],
            [0],
            color=fate_colors["absorbed"],
            lw=1.5,
            label=f"Absorbed ({A * 100:.1f}%)",
        ),
    ]
    leg = ax.legend(
        handles=legend_elements,
        loc="upper left",
        framealpha=0.15,
        facecolor="#1a1f2e",
        labelcolor="white",
        edgecolor="#333",
        title="Photon fate",
        prop={"size": 11, "weight": "bold"},
        title_fontproperties={"size": 12, "weight": "bold"},
    )
    leg.get_title().set_color("white")

    ax.text(
        x=0,
        y=padded_spread * 0.8,
        z=padded_spread * 0.85,
        s="Entry",
        color="#00E676",
        fontsize=8,
        ha="center",
        alpha=0.8,
    )
    ax.text(
        x=mc_model.z_max * 1000.0,
        y=padded_spread * 0.8,
        z=padded_spread * 0.85,
        s="Exit",
        color="#FFB300",
        fontsize=8,
        ha="center",
        alpha=0.8,
    )

    ax.view_init(elev=10, azim=-110)  # type: ignore
    ax.set_box_aspect(None, zoom=1.3)  # type: ignore

    # Resolve baseline filename without extension
    metadata = f"N{mc_model.num_photons}_mu{mc_model.mu_e}_Z{mc_model.z_max:.0f}km"
    base_filename = f"mc_scattering_3d_{metadata}"

    input_path = Path(filename) if filename is not None else None
    input_stem = input_path.stem if input_path is not None else base_filename
    input_suffix = input_path.suffix.lower() if input_path is not None else ""

    static_extensions = {".png", ".jpg", ".jpeg"}
    animation_extension = ".gif"

    # Optionally save the static figure first (saves as PNG inside PLOT_DIR)
    if save_figure:
        if input_path is None:
            figure_filename = f"{base_filename}.png"
        elif input_suffix in static_extensions:
            figure_filename = input_path.name
        else:
            figure_filename = f"{input_stem}.png"
            print(
                f"'{input_suffix}' extension is not valid for a static figure; "
                f"saving image as '{figure_filename}' instead."
            )

        PLOT_DIR.mkdir(parents=True, exist_ok=True)
        save_path = PLOT_DIR / figure_filename
        fig.savefig(
            save_path,
            dpi=300,
            facecolor=fig.get_facecolor(),
            bbox_inches="tight",
        )
        print(f"Static figure saved to: {save_path}")

    if animate:

        def update(angle):
            elevation = 10 + 5 * np.sin(np.radians(angle))  # subtle up-down motion
            ax.view_init(elev=elevation, azim=angle)  # type: ignore
            return (ax,)

        frames = np.arange(-120, 120, 2)
        anim = FuncAnimation(
            fig,
            update,
            frames=frames,
            interval=int(1000 / fps),
            blit=False,
            repeat=False,
        )

        if save_animation:
            if input_path is None:
                animation_filename = f"{base_filename}.gif"
            elif input_suffix == animation_extension:
                animation_filename = input_path.name
            else:
                animation_filename = f"{input_stem}.gif"
                print(
                    f"'{input_suffix}' extension is not valid for an animation; "
                    f"saving animation as '{animation_filename}' instead."
                )

            ANIMATION_DIR.mkdir(parents=True, exist_ok=True)
            filepath = ANIMATION_DIR / animation_filename
            anim.save(filepath, writer="ffmpeg", fps=fps, dpi=80)
            print(f"Animation is saved to: {filepath}")
            plt.close(fig)
        else:
            plt.show()

        return anim
    else:
        plt.show()
        return fig


mc_3d = PhotonTransportMC(
    mu_e_km1=20.0, albedo=0.9, g=0.85, z_max_km=0.2, num_photons=200
)

figure = visualize_mc_scattering_3d(
    mc_model=mc_3d,
    animate=True,
    save_figure=False,
    save_animation=True,
)

### **6.3 Animated Photon Transport**


Animations provide dynamic insight into step-by-step multiple scattering evolution across space. Below we define an animation wrapper that rotates the 3D perspective to reveal full 3D beam broadening.


In [ ]:
def animate_mc_scattering_step_by_step_3D(
    mc_model,
    fps=20,
    seed=42,
    save_animation=False,
    filename=None,
):
    # Set seed and run the simulation using the PhotonTransportMC class
    mc_model.seed = seed
    mc_model.analog_absorption = True
    paths, T, R, A = mc_model.run(return_weights=False, stop_at_boundary=True)

    z_max_m = mc_model.z_max * 1000.0
    print(
        f"Running 3D Monte Carlo Photon Transport Simulation for {mc_model.num_photons} "
        f"photons over {z_max_m:.0f} meters..."
    )

    print("Simulation Parameters:")
    print(f"  • μe={mc_model.mu_e}/km")
    print(f"  • g={mc_model.g}")
    print(f"  • ω={mc_model.albedo}")

    fate_colors = {
        "transmitted": "#4FC3F7",
        "reflected": "#EF5350",
        "absorbed": "#FFD54F",
    }

    processed = []
    max_spread = 0.0
    max_steps = 0

    # Process trajectories using original coordinates (in km) and convert to meters for plotting
    for path_x, path_y, path_z in paths:
        p_x = np.array(path_x) * 1000.0
        p_y = np.array(path_y) * 1000.0
        p_z = np.array(path_z) * 1000.0

        # Classify fate using static method from PhotonTransportMC
        fate = PhotonTransportMC.classify_trajectory(path_z, mc_model.z_max)

        processed.append((p_x, p_y, p_z, fate))
        max_steps = max(max_steps, len(p_z))
        max_spread = max(max_spread, np.max(np.abs(p_x)), np.max(np.abs(p_y)))

    max_spread = max(max_spread, 10.0)
    padded_spread = 1.25 * max_spread

    fig = plt.figure(figsize=(12, 9))
    fig.subplots_adjust(top=0.9, bottom=0.0, left=0.01, right=0.98)
    fig.patch.set_facecolor("#010408")
    ax = fig.add_subplot(111, projection="3d")
    ax.set_facecolor("#010408")

    # Entry and exit planes
    grid_pts = np.linspace(-padded_spread, padded_spread, 10)
    yy, xx = np.meshgrid(grid_pts, grid_pts)

    zz0 = np.zeros_like(xx)
    ax.plot_surface(zz0, xx, yy, color="#00E676", alpha=0.10, zorder=1)  # type: ignore

    zz_max = np.full_like(xx, z_max_m)
    ax.plot_surface(zz_max, xx, yy, color="#FFB300", alpha=0.10, zorder=1)  # type: ignore

    for i in range(len(grid_pts)):
        ax.plot(
            [0, 0],
            [grid_pts[i], grid_pts[i]],
            [grid_pts[0], grid_pts[-1]],
            color="#00E676",
            alpha=0.25,
            linewidth=0.5,
        )
        ax.plot(
            [0, 0],
            [grid_pts[0], grid_pts[-1]],
            [grid_pts[i], grid_pts[i]],
            color="#00E676",
            alpha=0.25,
            linewidth=0.5,
        )

        ax.plot(
            [z_max_m, z_max_m],
            [grid_pts[i], grid_pts[i]],
            [grid_pts[0], grid_pts[-1]],
            color="#FFB300",
            alpha=0.25,
            linewidth=0.5,
        )
        ax.plot(
            [z_max_m, z_max_m],
            [grid_pts[0], grid_pts[-1]],
            [grid_pts[i], grid_pts[i]],
            color="#FFB300",
            alpha=0.25,
            linewidth=0.5,
        )

    ax.set_xlim(-0.05 * z_max_m, 1.05 * z_max_m)
    ax.set_ylim(-padded_spread, padded_spread)
    ax.set_zlim(-padded_spread, padded_spread)  # type: ignore
    ax.set_axis_off()
    ax.set_box_aspect(None, zoom=1.25)  # type: ignore

    for pane in [ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane]:  # type: ignore
        pane.fill = False
        pane.set_edgecolor("#1a1f2e")

    ax.text(
        x=0,
        y=padded_spread * 0.8,
        z=padded_spread * 0.85,
        s="Entry",
        color="#00E676",
        fontsize=8,
        ha="center",
        alpha=0.8,
    )
    ax.text(
        x=z_max_m,
        y=padded_spread * 0.8,
        z=padded_spread * 0.85,
        s="Exit",
        color="#FFB300",
        fontsize=8,
        ha="center",
        alpha=0.8,
    )

    marker_ax = fig.add_axes((0.80, 0.83, 0.14, 0.14), facecolor="none")
    marker_ax.set_xlim(0.0, 1.0)
    marker_ax.set_ylim(0.0, 1.0)
    marker_ax.axis("off")

    marker_ax.annotate(
        "",
        xy=(0.9, 0.5),
        xytext=(0.25, 0.5),
        arrowprops=dict(arrowstyle="->", lw=2.0, color="#4FC3F7"),
    )
    marker_ax.text(
        0.93, 0.5, "Z", color="#4FC3F7", fontsize=9, fontweight="bold", va="center"
    )

    marker_ax.annotate(
        "",
        xy=(0.25, 0.9),
        xytext=(0.25, 0.5),
        arrowprops=dict(arrowstyle="->", lw=2.0, color="#EF5350"),
    )
    marker_ax.text(
        0.23, 0.93, "X", color="#EF5350", fontsize=9, fontweight="bold", ha="right"
    )

    marker_ax.annotate(
        "",
        xy=(0.58, 0.2),
        xytext=(0.25, 0.5),
        arrowprops=dict(arrowstyle="->", lw=2.0, color="#00E676"),
    )
    marker_ax.text(
        0.61, 0.18, "Y", color="#00E676", fontsize=9, fontweight="bold", va="top"
    )

    legend_items = [
        Line2D(
            [0],
            [0],
            color=fate_colors["transmitted"],
            lw=1.6,
            label=f"Transmitted ({T * 100:.1f}%)",
        ),
        Line2D(
            [0],
            [0],
            color=fate_colors["reflected"],
            lw=1.6,
            label=f"Reflected ({R * 100:.1f}%)",
        ),
        Line2D(
            [0],
            [0],
            color=fate_colors["absorbed"],
            lw=1.6,
            label=f"Absorbed ({A * 100:.1f}%)",
        ),
    ]
    leg = fig.legend(
        handles=legend_items,
        loc="upper left",
        framealpha=0.15,
        facecolor="#020075",
        edgecolor="#566573",
        title="Photon fate",
        prop={"size": 10, "weight": "bold"},
        title_fontproperties={"size": 11, "weight": "bold"},
    )
    leg.get_title().set_color("white")
    for t in leg.get_texts():
        t.set_color("white")

    fig_title = (
        f"3D Monte Carlo Photon Transport\n"
        f"$N$={mc_model.num_photons}  •  $\\mu_e$={mc_model.mu_e}/km  •  $g$={mc_model.g}  •  "
        f"$\\omega$={mc_model.albedo}  •  $Z$={z_max_m:.0f}m"
    )
    fig.suptitle(fig_title, color="white", y=0.96)

    base_elev = 10
    base_azim = -110
    ax.view_init(elev=base_elev, azim=base_azim)  # type: ignore
    ax.set_box_aspect(None, zoom=1.3)  # type: ignore

    lines = []
    heads = []
    for _, _, _, fate in processed:
        (ln,) = ax.plot([], [], [], color=fate_colors[fate], lw=0.9, alpha=0.85)
        head_marker = "s" if fate in ("transmitted", "reflected") else "o"
        hd = ax.scatter(
            [],
            [],
            [],
            color=fate_colors[fate],
            s=14,
            marker=head_marker,
            depthshade=False,
            edgecolors="white",
            linewidths=0.4,
        )
        lines.append(ln)
        heads.append(hd)

    def init():
        for ln in lines:
            ln.set_data([], [])
            ln.set_3d_properties([])
        return tuple(lines)

    def update(frame):
        for i, (p_x, p_y, p_z, _) in enumerate(processed):
            end_idx = min(frame + 1, len(p_z))
            lines[i].set_data(p_z[:end_idx], p_x[:end_idx])
            lines[i].set_3d_properties(p_y[:end_idx])

            if end_idx > 0:
                heads[i]._offsets3d = (
                    [p_z[end_idx - 1]],
                    [p_x[end_idx - 1]],
                    [p_y[end_idx - 1]],
                )

        azim = base_azim + 0.6 * frame
        elev = base_elev + 3.0 * np.sin(np.radians(frame * 2.0))
        ax.view_init(elev=elev, azim=azim)  # type: ignore
        ax.set_box_aspect(None, zoom=1.3)  # type: ignore

        return tuple(lines)

    anim = FuncAnimation(
        fig,
        update,
        init_func=init,
        frames=max_steps,
        interval=int(1000 / fps),
        blit=False,
        repeat=False,
    )

    if save_animation:
        if filename is None:
            filename = f"mc_scattering_step_mue_e_{mc_model.mu_e}_N{mc_model.num_photons}_z_{z_max_m:.0f}"

        # Ensure correct extension is appended
        if not filename.endswith(".gif"):
            filename = f"{filename}.gif"

        print("Saving the animation...")
        ANIMATION_DIR.mkdir(parents=True, exist_ok=True)
        out_path = ANIMATION_DIR / filename
        anim.save(out_path, writer="ffmpeg", fps=fps, dpi=100)
        print(f"Animation saved to: {out_path}")

        plt.close(fig)
    else:
        plt.show()

    return anim


# Execution call using the class instance
mc_step_by_step = PhotonTransportMC(
    mu_e_km1=20.0,
    albedo=0.9,
    g=0.85,
    z_max_km=0.15,
    num_photons=500,
)

animation = animate_mc_scattering_step_by_step_3D(
    mc_model=mc_step_by_step,
    fps=10,
    seed=42,
    save_animation=True,
)

### **6.4 Propagation-Depth Comparison**


Comparing photon trajectories across increasing fog layer thicknesses ($Z \in \{100, 250, 500\}\,\text{m}$) illustrates how transmission exponentially degrades while backscattering and lateral spreading dominate. It can be also observed that as the fog layer thickness increases more number of photons are absorbed and the transmitted photons are reduced. The lateral spread of the beam also increases with increasing fog thickness, indicating that multiple scattering events are more likely to occur in thicker fog layers.


#### **6.4.1 Depth Comparison in 2D**



Below we plot the 2D projection of photon trajectories for three different fog layer thicknesses, highlighting the effects of multiple scattering and absorption on beam propagation. The fog region is shaded in light gray to indicate the extent of the scattering medium. The extent of the depth can be tweaked by adjusting the `depths_m` parameter in the function call. The plot also displays the percentage of transmitted and reflected photons for each depth, providing a clear visual representation of how fog thickness impacts photon transport.

In [ ]:
def plot_propagation_depth_comparison(
    mu_e=20.0,
    albedo=0.9,
    g=0.85,
    depths_m=[100, 250, 500],
    num_photons=200,
    save=False,
    filename=None,
):
    fig, axes = plt.subplots(1, len(depths_m), figsize=(14, 5), sharey=True)
    fig.suptitle(
        "Monte Carlo Trajectory Comparison Across Propagation Depths",
    )

    for i, z_m in enumerate(depths_m):
        mc = PhotonTransportMC(
            mu_e, albedo, g, z_m / 1000.0, num_photons=num_photons, seed=42
        )
        paths, T, R, A = mc.run(stop_at_boundary=True)
        ax = axes[i]
        for px, py, pz in paths:
            ax.plot(np.array(pz) * 1000.0, np.array(px) * 1000.0, alpha=0.6, lw=1)
        ax.axvline(0, color="black", linestyle="--", label="Entrance")
        ax.axvline(z_m, color="red", linestyle="--", label=f"Exit ({z_m}m)")
        # filling the FOG  region
        ax.axvspan(0, z_m, color="lightgray", alpha=0.6, zorder=0)

        ax.set_xlabel("Propagation Depth Z (m)")
        if i == 0:
            ax.set_ylabel("Transverse Spread X (m)")
        ax.set_title(
            f"Z = {z_m} m\nT={T * 100:.1f}%, R={R * 100:.1f}%, A={A * 100:.1f}%"
        )
        ax.legend(loc="upper center")

    plt.tight_layout()
    if save:
        if filename is not None and not filename.endswith((".png", ".jpg", ".jpeg")):
            filename += ".png"
        else:
            filename = "propagation_depth_comparison.png"

        PLOT_DIR.mkdir(parents=True, exist_ok=True)
        save_path = PLOT_DIR / filename
        plt.savefig(save_path, dpi=300)
        print(f"Figure saved to {save_path}")
    plt.show()


plot_propagation_depth_comparison(save=True)


#### **6.4.2 Depth Comparison in 3D**


In the next code cell below we demonstrated the effect of the Fog layer thickness on the trajectories of the photons in 3D. The fog region is shaded in light gray to indicate the extent of the scattering medium. The extent of the depth can be tweaked by adjusting the `depths_m` parameter in the function call. The plot also displays the percentage of transmitted and reflected photons for each depth, providing a clear visual representation of how fog thickness impacts photon transport. As observed in the figure, the lateral spread of the beam increases with increasing fog thickness, indicating that multiple scattering events are more likely to occur in thicker fog layers. The amount of photons absorbed also increases with increasing fog thickness, leading to a decrease in the number of transmitted photons. This highlights the importance of considering fog thickness in applications such as free-space optical communications and LiDAR systems, where signal attenuation and scattering can significantly impact performance.

In [ ]:
def propagation_depth_comparison_3d(
    mc_model,
    depths_m=(50, 100, 250, 500),
    fps=15,
    seed=42,
    save_figure=False,
    animate=False,
    save_animation=False,
    filename=None,
):
    if len(depths_m) != 4:
        raise ValueError("Please provide exactly 4 values for a 2x2 grid.")

    fate_colors = {
        "transmitted": "#4FC3F7",
        "reflected": "#EF5350",
        "absorbed": "#FFD54F",
    }

    print(f"Starting 3D Monte Carlo simulation with {mc_model.num_photons} photons...")
    print("Simulation Parameters:")
    print(f"  • Extinction Coefficient: {mc_model.mu_e}/km")
    print(f"  • Asymmetry Parameter: {mc_model.g}")
    print(f"  • Albedo: {mc_model.albedo}")
    mc_model.seed = seed
    mc_model.analog_absorption = True
    all_results = {}
    for z_max_m in depths_m:
        z_max_km = z_max_m / 1000.0
        # update the z_max of the model for each depth
        mc_model.z_max = z_max_km
        paths, T, R, A = mc_model.run(return_weights=False, stop_at_boundary=True)
        print(
            f"Simulation results for z_max={z_max_m} m: T={T:.4f}, R={R:.4f}, A={A:.4f}"
        )
        all_results[z_max_m] = (paths, T, R, A)

    fig = plt.figure(figsize=(14, 10))
    fig.patch.set_facecolor("#010408")

    left, right, bottom, top = 0.03, 0.99, 0.04, 0.86
    fig.subplots_adjust(
        left=left, right=right, bottom=bottom, top=top, wspace=0.03, hspace=0.08
    )

    title_str = (
        f"3D Monte Carlo Photon Transport Comparison\n"
        f"N={mc_model.num_photons}  •  $\\mu_e$={mc_model.mu_e}/km  •  g={mc_model.g}  •  $\\omega$={mc_model.albedo}"
    )
    fig.suptitle(title_str, color="white", y=0.965)

    axes = []
    for i, (z_max_m, (paths, T, R, A)) in enumerate(all_results.items(), start=1):
        ax = fig.add_subplot(2, 2, i, projection="3d")
        ax.set_facecolor("#010408")
        axes.append(ax)

        max_spread = 0.0
        n_scatter_list = []

        for path_x, path_y, path_z in paths:
            p_x = np.array(path_x) * 1000.0
            p_y = np.array(path_y) * 1000.0
            current_max = max(np.max(np.abs(p_x)), np.max(np.abs(p_y)))
            max_spread = max(max_spread, current_max)
            n_scatter_list.append(len(path_x) - 1)

        max_spread = max(max_spread, 10.0)
        padded_spread = max_spread * 1.25
        max_n = max(n_scatter_list) if n_scatter_list else 1

        for (path_x, path_y, path_z), n_scatter in zip(paths, n_scatter_list):
            p_x = np.array(path_x) * 1000.0
            p_y = np.array(path_y) * 1000.0
            p_z = np.array(path_z) * 1000.0

            # Classifying the fate of the photon trajectory
            fate = PhotonTransportMC.classify_trajectory(p_z, z_max_m)
            base_color = fate_colors[fate]

            alpha = max(0.15, 0.8 - 0.6 * (n_scatter / max_n))
            lw = max(0.4, 1.2 - 0.8 * (n_scatter / max_n))

            ax.plot(
                p_z,
                p_x,
                p_y,
                color=base_color,
                alpha=alpha if fate != "absorbed" else 0.85,
                linewidth=lw if fate != "absorbed" else 0.7,
                zorder=2 if fate != "absorbed" else 3,
            )

            if fate == "absorbed":
                ax.scatter(
                    [p_z[-1]],
                    [p_x[-1]],
                    [p_y[-1]],
                    color=base_color,
                    s=14,
                    zorder=6,
                    depthshade=False,
                    edgecolors="white",
                    linewidths=0.35,
                )
            else:
                ax.scatter(
                    [p_z[-1]],
                    [p_x[-1]],
                    [p_y[-1]],
                    color=base_color,
                    s=14,
                    marker="s",
                    zorder=6,
                    depthshade=False,
                    edgecolors="white",
                    linewidths=0.35,
                )

        # Entry / Exit planes
        grid_pts = np.linspace(-padded_spread, padded_spread, 10)
        yy, xx = np.meshgrid(grid_pts, grid_pts)

        zz0 = np.zeros_like(xx)
        ax.plot_surface(zz0, xx, yy, color="#00E676", alpha=0.10, zorder=1)  # type: ignore

        for j in range(len(grid_pts)):
            ax.plot(
                [0, 0],
                [grid_pts[j], grid_pts[j]],
                [grid_pts[0], grid_pts[-1]],
                color="#00E676",
                alpha=0.25,
                linewidth=0.5,
            )
            ax.plot(
                [0, 0],
                [grid_pts[0], grid_pts[-1]],
                [grid_pts[j], grid_pts[j]],
                color="#00E676",
                alpha=0.25,
                linewidth=0.5,
            )

        zz_max = np.full_like(xx, z_max_m)
        ax.plot_surface(zz_max, xx, yy, color="#FFB300", alpha=0.10, zorder=1)  # type: ignore

        for j in range(len(grid_pts)):
            ax.plot(
                [z_max_m, z_max_m],
                [grid_pts[j], grid_pts[j]],
                [grid_pts[0], grid_pts[-1]],
                color="#FFB300",
                alpha=0.25,
                linewidth=0.5,
            )
            ax.plot(
                [z_max_m, z_max_m],
                [grid_pts[0], grid_pts[-1]],
                [grid_pts[j], grid_pts[j]],
                color="#FFB300",
                alpha=0.25,
                linewidth=0.5,
            )

        ax.set_xlim(-0.05 * z_max_m, z_max_m * 1.05)
        ax.set_ylim(-padded_spread, padded_spread)
        ax.set_zlim(-padded_spread, padded_spread)  # type: ignore
        ax.set_axis_off()
        ax.set_box_aspect(None, zoom=1.15)  # type: ignore

        for pane in [ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane]:  # type: ignore
            pane.fill = False
            pane.set_edgecolor("#1a1f2e")

        ax.text(
            x=0,
            y=padded_spread * 0.8,
            z=padded_spread * 0.85,
            s="Entry",
            color="#00E676",
            fontsize=7.5,
            ha="center",
            alpha=0.8,
        )
        ax.text(
            x=z_max_m,
            y=padded_spread * 0.8,
            z=padded_spread * 0.85,
            s="Exit",
            color="#FFB300",
            fontsize=7.5,
            ha="center",
            alpha=0.8,
        )

        ax.text2D(  # type: ignore
            0.5,
            0.96,
            f"Z={z_max_m} m\nT={T * 100:.1f}%, R={R * 100:.1f}%, A={A * 100:.1f}%",
            transform=ax.transAxes,
            ha="center",
            va="top",
            color="white",
            fontsize=12,
            fontweight="bold",
        )

    legend_handles = [
        Line2D([0], [0], color=fate_colors["transmitted"], lw=1.8, label="Transmitted"),
        Line2D([0], [0], color=fate_colors["reflected"], lw=1.8, label="Reflected"),
        Line2D([0], [0], color=fate_colors["absorbed"], lw=1.8, label="Absorbed"),
    ]
    leg = fig.legend(
        handles=legend_handles,
        loc="upper left",
        bbox_to_anchor=(0.012, 0.985),
        framealpha=0.25,
        facecolor="#020075",
        edgecolor="#566573",
        title="Photon fate",
        prop={"size": 10, "weight": "bold"},
        title_fontproperties={"size": 11, "weight": "bold"},
    )
    leg.get_title().set_color("white")
    for t in leg.get_texts():
        t.set_color("white")

    marker_ax = fig.add_axes((0.88, 0.85, 0.10, 0.13), facecolor="none")
    marker_ax.set_xlim(0.0, 1.0)
    marker_ax.set_ylim(0.0, 1.0)
    marker_ax.axis("off")

    marker_ax.annotate(
        "",
        xy=(0.9, 0.5),
        xytext=(0.25, 0.5),
        arrowprops=dict(arrowstyle="->", lw=2.0, color="#4FC3F7"),
    )
    marker_ax.text(
        0.93, 0.5, "Z", color="#4FC3F7", fontsize=9, fontweight="bold", va="center"
    )

    marker_ax.annotate(
        "",
        xy=(0.25, 0.9),
        xytext=(0.25, 0.5),
        arrowprops=dict(arrowstyle="->", lw=2.0, color="#EF5350"),
    )
    marker_ax.text(
        0.23, 0.93, "X", color="#EF5350", fontsize=9, fontweight="bold", ha="right"
    )

    marker_ax.annotate(
        "",
        xy=(0.58, 0.2),
        xytext=(0.25, 0.5),
        arrowprops=dict(arrowstyle="->", lw=2.0, color="#00E676"),
    )
    marker_ax.text(
        0.61, 0.18, "Y", color="#00E676", fontsize=9, fontweight="bold", va="top"
    )

    border_color = "#263241"
    outer = Rectangle(
        (left, bottom),
        right - left,
        top - bottom,
        transform=fig.transFigure,
        fill=False,
        linewidth=1.2,
        edgecolor=border_color,
        zorder=20,
    )
    fig.add_artist(outer)

    mid_x = 0.5 * (left + right)
    mid_y = 0.5 * (bottom + top)

    fig.lines.append(
        Line2D(
            [mid_x, mid_x],
            [bottom, top],
            transform=fig.transFigure,
            color=border_color,
            linewidth=1.0,
            zorder=20,
        )
    )
    fig.lines.append(
        Line2D(
            [left, right],
            [mid_y, mid_y],
            transform=fig.transFigure,
            color=border_color,
            linewidth=1.0,
            zorder=20,
        )
    )

    for ax in axes:
        ax.view_init(elev=10, azim=-110)
        ax.set_box_aspect(None, zoom=1.15)

    # Save the static figure if requested
    base_filename = f"mc_scattering_3d_2x2_N{mc_model.num_photons}_mue_{mc_model.mu_e}"

    input_path = Path(filename) if filename is not None else None
    input_stem = input_path.stem if input_path is not None else base_filename
    input_suffix = input_path.suffix.lower() if input_path is not None else ""

    static_extensions = {".png", ".jpg", ".jpeg"}
    animation_extension = ".gif"

    if save_figure:
        if input_path is None:
            figure_filename = f"{base_filename}.png"
        elif input_suffix in static_extensions:
            figure_filename = input_path.name
        else:
            figure_filename = f"{input_stem}.png"
            print(
                f"'{input_suffix}' extension is not valid for a static figure; "
                f"saving image as '{figure_filename}' instead."
            )

        PLOT_DIR.mkdir(parents=True, exist_ok=True)
        save_path = PLOT_DIR / figure_filename
        fig.savefig(
            save_path,
            dpi=300,
            facecolor=fig.get_facecolor(),
            bbox_inches="tight",
        )
        print(f"Static figure saved to: {save_path}")

    # Rotate the 3D view for all subplots if animation is requested
    if animate:
        # Synchronous rotation for all subplots
        def update(angle):
            elevation = 10 + 5 * np.sin(np.radians(angle))
            for ax in axes:
                ax.view_init(elev=elevation, azim=angle)
            return tuple(axes)

        frames = np.arange(-120, 240)
        anim = FuncAnimation(
            fig,
            update,
            frames=frames,
            interval=int(1000 / fps),
            blit=False,
            repeat=False,
        )

        if save_animation:
            if input_path is None:
                animation_filename = f"{base_filename}.gif"
            elif input_suffix == animation_extension:
                animation_filename = input_path.name
            else:
                animation_filename = f"{input_stem}.gif"
                print(
                    f"'{input_suffix}' extension is not valid for an animation; "
                    f"saving animation as '{animation_filename}' instead."
                )

            print("Saving the animation...")
            ANIMATION_DIR.mkdir(parents=True, exist_ok=True)
            filepath = ANIMATION_DIR / animation_filename
            anim.save(filepath, writer="ffmpeg", fps=fps)
            print(f"Animation is saved to: {filepath}")
            plt.close(fig)
        else:
            plt.show()

        return anim

    else:
        plt.show()
        return fig


# Initialize a PhotonTransportMC instance for the 3D comparison
mc_comparison = PhotonTransportMC(
    mu_e_km1=20.0,
    albedo=0.9,
    g=0.85,
    z_max_km=0.2,
    num_photons=200,
)

figure = propagation_depth_comparison_3d(
    mc_model=mc_comparison,
    depths_m=(50, 100, 250, 500),
    fps=20,
    seed=42,
    save_figure=True,
    animate=True,
    save_animation=True,
    filename="mc_scattering_3d_2x2_depth_comparison.gif",
)

## 7. Angular Scattering Intensity


### **7.1 Intensity Binning and Solid-Angle Normalization**

When photons exit the fog layer—either by penetrating the forward boundary ($z \ge z_{\max}$) or by reflecting backward out of the entrance plane ($z \le 0$)—their individual escape angles and accumulated statistical weights must be converted into a continuous, macroscopic angular intensity distribution $I(\theta)$. 

This conversion, encapsulated in `PhotonTransportMC.calculate_intensity()`, is executed in three rigorous steps:


#### **Angular Binning and Weight Accumulation**
First, the polar escape angle range $[0, \pi/2]$ ($0^\circ$ to $90^\circ$) relative to the boundary normal is uniformly partitioned into $N_{\text{bins}}$ (`num_bins`) angular intervals of constant width:

$$\Delta \theta = \frac{\pi / 2}{N_{\text{bins}}}$$

Each angular bin $i$ is centered at polar angle $\theta_i = \left(\theta_{i, \text{start}} + \theta_{i, \text{end}}\right) / 2$. Using `np.digitize`, every surviving photon $j$ exiting with polar angle $\theta_j$ and statistical weight $w_j$ is mapped to its corresponding angular bin $i$. The total optical energy (or statistical weight) exiting into bin $i$ is summed across all matched trajectories:

$$W_i = \sum_{j \in \text{bin } i} w_j$$


#### **Solid-Angle Geometry and Conical Ring Expansion**
Simply plotting the raw weight sum $W_i$ against polar angle $\theta_i$ would produce a severely distorted physical picture. In 3D spherical coordinates, an angular bin bounded between $\theta_i - \Delta\theta/2$ and $\theta_i + \Delta\theta/2$, rotated over the full $2\pi$ azimuth $\phi$, defines a **conical ring (annular zone)** on the unit sphere.

The exact differential solid angle $\Delta \Omega_i$ (in steradians, $\text{sr}$) subtended by this conical ring is obtained by surface integration:

$$\Delta \Omega_i = \int_0^{2\pi} d\phi \int_{\theta_i - \frac{\Delta\theta}{2}}^{\theta_i + \frac{\Delta\theta}{2}} \sin \theta \, d\theta = 2\pi \left[ \cos\left(\theta_i - \frac{\Delta\theta}{2}\right) - \cos\left(\theta_i + \frac{\Delta\theta}{2}\right) \right]$$

Applying the trigonometric identity for differences of cosines and taking the first-order Taylor expansion (which is exact as $\Delta \theta \to 0$), our code evaluates the differential solid angle directly as:

$$\Delta \Omega_i \approx 2\pi \sin \theta_i \, \Delta \theta$$

#### **Why Solid-Angle Normalization is Critical ($\sin \theta_i$ dependence)**
Notice that the solid angle $\Delta \Omega_i$ scales directly with $\sin \theta_i$:
* **Near the Forward Axis ($\theta_i \to 0^\circ$):** The differential solid angle $\Delta \Omega_i \to 0$. Because the conical ring near the pole has almost zero surface area, even a small total photon weight ($W_i$) exiting near the optical axis represents an **extremely high power density per steradian** (a tightly focused, high-intensity laser beam).
* **At Wide Angles ($\theta_i \to 90^\circ$):** The conical ring near the equator spans a vast surface area ($\sin \theta_i \to 1$). Photons exiting at wide angles are dispersed across a much larger solid angle, diluting the radiometric intensity per steradian.

Without dividing by $\Delta \Omega_i$, the artificial geometric expansion of solid angles at wide angles would misleadingly make isotropic or wide-angle scattering appear much stronger than axial transmission.


#### **Normalized Radiometric Scattering Intensity Equation**
To obtain the normalized scattering intensity per steradian ($\text{sr}^{-1}$), the accumulated bin weight $W_i$ is divided by both the total number of simulated incident photons ($N_{\text{total}} = \text{num\_photons}$) and the differential solid angle $\Delta \Omega_i$:

$$I(\theta_i) = \frac{W_i}{N_{\text{total}} \cdot \Delta \Omega_i} = \frac{\sum_{j \in \text{bin } i} w_j}{N_{\text{total}} \cdot 2\pi \sin \theta_i \, \Delta \theta}$$

#### **Physical Interpretation & Conservation Check**
* **Units:** $I(\theta_i)$ represents the **probability density of photon escape per unit solid angle** ($\text{sr}^{-1}$).
* **Hemispherical Integration:** Integrating $I(\theta)$ over the entire forward or backward hemisphere exactly recovers the macroscopic diffuse transmission ($T_{\text{diffuse}}$) or diffuse reflection ($R_{\text{diffuse}}$) fraction:

$$T_{\text{diffuse}} = \int_{\text{forward}} I(\theta_f) \, d\Omega = \int_0^{\pi/2} I(\theta_f) \cdot 2\pi \sin \theta_f \, d\theta_f$$

$$R_{\text{diffuse}} = \int_{\text{backward}} I(\theta_b) \, d\Omega = \int_0^{\pi/2} I(\theta_b) \cdot 2\pi \sin \theta_b \, d\theta_b$$

#### **Numerical Singularity Protection**
Because $\sin(0^\circ) = 0$, evaluating $I(\theta_i)$ exactly at $\theta_0 = 0$ could trigger a division-by-zero (`ZeroDivisionError`). In `calculate_intensity()`, bin centers $\theta_i$ are evaluated at the exact midpoint (`(bin_edges[:-1] + bin_edges[1:]) / 2.0`), ensuring $\theta_i > 0$. As an added safety barrier against floating-point underflow, the denominator is safeguarded with `max(1e-12, total_photons * solid_angle)`.

In [ ]:
def save_intensity_dataset(path, dataset, metadata=None):
    """Caches the dataset as compressed NPZ and outputs parameter metadata as JSON."""
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, **dataset)
    if metadata is not None:
        with open(path.with_suffix(".json"), "w") as f:
            json.dump(metadata, f, indent=4)
    print(f"Dataset cached successfully to {path}")


def load_intensity_dataset(path):
    """Loads and returns the cached dataset files as a dictionary."""
    data = np.load(path)
    print(f"Dataset loaded from cache: {path}")
    return {k: data[k] for k in data.files}


def make_serializable(obj):
    """Recursively converts any nested data structures to JSON-serializable types."""
    if isinstance(obj, dict):
        return {str(k): make_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple, set)):
        return [make_serializable(x) for x in obj]
    elif isinstance(obj, np.ndarray):
        return f"ndarray(shape={obj.shape}, dtype={obj.dtype})"
    elif callable(obj):
        return f"function: {getattr(obj, '__name__', str(obj))}"
    elif isinstance(obj, complex):
        return f"{obj.real} + {obj.imag}j"
    elif isinstance(obj, (int, float, str, bool)) or obj is None:
        return obj
    else:
        return str(obj)


def serialize_metadata_for_json(sweep_type, param_values, fixed_params):
    """Safely converts all simulation parameters to a JSON-serializable dictionary."""
    meta = {
        "sweep_type": sweep_type,
        "param_values": [make_serializable(v) for v in param_values]
        if isinstance(param_values, (list, np.ndarray))
        else make_serializable(param_values),
    }

    fixed_clean = {}
    for k, v in fixed_params.items():
        if k == "metadata":
            if isinstance(v, dict):
                for sub_k, sub_v in v.items():
                    fixed_clean[sub_k] = make_serializable(sub_v)
            continue

        fixed_clean[k] = make_serializable(v)

    meta["fixed_params"] = fixed_clean
    return meta


def get_parameter_specific_path(sweep_type, param_values, fixed_params, base_path):
    """
    Constructs a unique filename by embedding key parameter scalars and
    a deterministic parameter hash directly into the path.
    """
    base_path = Path(base_path)
    parent_dir = base_path.parent
    base_stem = base_path.stem

    parts = [base_stem]

    # Format and append key scalar fixed parameters
    for k in sorted(fixed_params.keys()):
        if k in ["metadata", "fog_models", "r_grid", "ref_indices"]:
            continue
        v = fixed_params[k]
        if isinstance(v, (int, float, str)):
            # Format float nicely without trailing zeros or unnecessary decimals
            if isinstance(v, float):
                v_str = f"{v:.4f}".rstrip("0").rstrip(".")
            else:
                v_str = str(v)

            if k == "num_photons":
                clean_k = "N"
            else:
                clean_k = "".join(c for c in k if c.isalnum()).lower()
            parts.append(f"{clean_k}_{v_str}")

    # Format and append sweep values
    val_strs = []
    for v in param_values:
        if isinstance(v, float):
            val_strs.append(f"{v:.4f}".rstrip("0").rstrip("."))
        else:
            val_strs.append(str(v))
    parts.append("vals_" + "_".join(val_strs))

    # Create a hash of all inputs (including numpy arrays, dicts, and functions)
    hasher = hashlib.md5()
    hasher.update(str(sweep_type).encode())
    hasher.update(str(param_values).encode())
    for k in sorted(fixed_params.keys()):
        if k == "metadata":
            continue
        val = fixed_params[k]
        if isinstance(val, np.ndarray):
            hasher.update(val.tobytes())
        elif isinstance(val, dict):
            hasher.update(str(sorted(val.items(), key=lambda x: str(x[0]))).encode())
        elif callable(val):
            hasher.update(val.__name__.encode())
        else:
            hasher.update(str(val).encode())

    param_hash = hasher.hexdigest()[:8]
    parts.append(f"hash_{param_hash}")

    filename = "_".join(parts) + ".npz"
    return parent_dir / filename


def get_metadata_plot_path(
    save_filename, sweep_type, param_values, fixed_params, plot_dir
):
    """
    Constructs a unique plot filename by appending key parameter metadata and hash.
    """
    if not save_filename:
        return None

    save_filename = Path(save_filename)
    base_stem = save_filename.stem
    extension = save_filename.suffix

    parts = [base_stem]

    if fixed_params:
        # Format and append key scalar fixed parameters
        for k in sorted(fixed_params.keys()):
            if k in ["metadata", "fog_models", "r_grid", "ref_indices"]:
                continue
            v = fixed_params[k]
            if isinstance(v, (int, float, str)):
                if isinstance(v, float):
                    v_str = f"{v:.4f}".rstrip("0").rstrip(".")
                else:
                    v_str = str(v)

                if k == "num_photons":
                    clean_k = "N"
                else:
                    clean_k = "".join(c for c in k if c.isalnum()).lower()
                parts.append(f"{clean_k}_{v_str}")

        # Format and append sweep values
        val_strs = []
        for v in param_values:
            if isinstance(v, float):
                val_strs.append(f"{v:.4f}".rstrip("0").rstrip("."))
            else:
                val_strs.append(str(v))
        parts.append("vals_" + "_".join(val_strs))

        # Append short MD5 hash
        hasher = hashlib.md5()
        hasher.update(str(sweep_type).encode())
        hasher.update(str(param_values).encode())
        for k in sorted(fixed_params.keys()):
            if k == "metadata":
                continue
            val = fixed_params[k]
            if isinstance(val, np.ndarray):
                hasher.update(val.tobytes())
            elif isinstance(val, dict):
                hasher.update(
                    str(sorted(val.items(), key=lambda x: str(x[0]))).encode()
                )
            elif callable(val):
                hasher.update(val.__name__.encode())
            else:
                hasher.update(str(val).encode())

        param_hash = hasher.hexdigest()[:8]
        parts.append(f"hash_{param_hash}")

    filename = "_".join(parts) + extension
    return Path(plot_dir) / filename


def load_and_normalize_legacy_keys(path, sweep_type):
    """Loads a dataset and normalizes any generic 'param_' key prefixes to the sweep type."""
    raw_data = load_intensity_dataset(path)
    normalized = {}
    for k, v in raw_data.items():
        if k.startswith("param_"):
            new_key = k.replace("param_", f"{sweep_type}_", 1)
            normalized[new_key] = v
        else:
            normalized[k] = v
    return normalized


def run_or_load_intensity_sweep(
    sweep_type,
    param_values,
    fixed_params,
    data_path,
    force_run=False,
    smooth_window=11,
    smooth_poly=2,
):
    actual_data_path = get_parameter_specific_path(
        sweep_type, param_values, fixed_params, data_path
    )

    # Check if the parameter-specific dataset already exists
    if actual_data_path.exists() and not force_run:
        return load_and_normalize_legacy_keys(actual_data_path, sweep_type)

    # Fallback: Check if the legacy base filename exists from a previous run
    base_path = Path(data_path)
    if base_path.exists() and not force_run:
        print(f"\n[CACHE HITTING] Found historical cache file at {base_path}.")
        print(
            "Migrating dataset and normalising keys to the parameter-specific caching directory..."
        )
        dataset = load_and_normalize_legacy_keys(base_path, sweep_type)
        full_metadata = serialize_metadata_for_json(
            sweep_type, param_values, fixed_params
        )
        save_intensity_dataset(actual_data_path, dataset, full_metadata)
        return dataset

    # Cache Miss: Run the full simulation
    dataset = {}
    mu_e = fixed_params.get("mu_e", 39.12)
    z_max = fixed_params.get("z_max", 0.05)
    num_photons = fixed_params.get("num_photons", 500000)

    if sweep_type == "albedo":
        g_fixed = fixed_params["g"]
        for w0 in param_values:
            print(f"Simulating albedo w0 = {w0}...")
            mc = PhotonTransportMC(
                mu_e,
                w0,
                g_fixed,
                z_max,
                num_photons=num_photons,
                analog_absorption=False,
            )
            f_ang, f_wgt, b_ang, b_wgt = mc.run(return_weights=True)
            deg_f, int_f = PhotonTransportMC.calculate_intensity(
                f_ang, f_wgt, num_photons
            )
            deg_b, int_b = PhotonTransportMC.calculate_intensity(
                b_ang, b_wgt, num_photons
            )
            if smooth_window and len(int_b) >= smooth_window:
                int_b = savgol_filter(int_b, smooth_window, smooth_poly, mode="interp")
            dataset[f"albedo_{w0}_forward_angle_deg"] = deg_f
            dataset[f"albedo_{w0}_forward_intensity"] = int_f
            dataset[f"albedo_{w0}_backward_angle_deg"] = deg_b
            dataset[f"albedo_{w0}_backward_intensity"] = int_b

    elif sweep_type == "g":
        w0_fixed = fixed_params["albedo"]
        for g_val in param_values:
            print(f"Simulating asymmetry factor g = {g_val}...")
            mc = PhotonTransportMC(
                mu_e,
                w0_fixed,
                g_val,
                z_max,
                num_photons=num_photons,
                analog_absorption=False,
            )
            f_ang, f_wgt, b_ang, b_wgt = mc.run(return_weights=True)
            deg_f, int_f = PhotonTransportMC.calculate_intensity(
                f_ang, f_wgt, num_photons
            )
            deg_b, int_b = PhotonTransportMC.calculate_intensity(
                b_ang, b_wgt, num_photons
            )
            if smooth_window and len(int_f) >= smooth_window:
                int_f = savgol_filter(int_f, smooth_window, smooth_poly, mode="interp")
            if smooth_window and len(int_b) >= smooth_window:
                int_b = savgol_filter(int_b, smooth_window, smooth_poly, mode="interp")
            dataset[f"g_{g_val}_forward_angle_deg"] = deg_f
            dataset[f"g_{g_val}_forward_intensity"] = int_f
            dataset[f"g_{g_val}_backward_angle_deg"] = deg_b
            dataset[f"g_{g_val}_backward_intensity"] = int_b

    elif sweep_type == "backscatter_wavelength":
        vis = fixed_params["visibility_km"]
        fog_models = fixed_params["fog_models"]
        r_grid = fixed_params["r_grid"]
        ref_indices = fixed_params["ref_indices"]
        for fog_name, n_func in fog_models.items():
            prefix = fog_name.lower().replace(" ", "_")
            print(f"--- Simulating Backscatter Sensitivity for {fog_name} ---")
            for wl in param_values:
                m = ref_indices[wl]
                Qe_arr, Qs_arr, g_arr = (
                    np.zeros_like(r_grid),
                    np.zeros_like(r_grid),
                    np.zeros_like(r_grid),
                )
                for i, r in enumerate(r_grid):
                    Qe, Qs, _, g_val = mie.efficiencies(m, 2 * r, wl)
                    Qe_arr[i], Qs_arr[i], g_arr[i] = Qe, Qs, g_val
                weight = (r_grid**2) * n_func(r_grid, vis)
                mu_e_km1 = np.pi * trapezoid(weight * Qe_arr, r_grid) * 1e-12 * 1000.0
                mu_s_km1 = np.pi * trapezoid(weight * Qs_arr, r_grid) * 1e-12 * 1000.0
                w0 = mu_s_km1 / mu_e_km1 if mu_e_km1 > 0 else 0.0
                g_avg = (
                    trapezoid(weight * Qs_arr * g_arr, r_grid)
                    / trapezoid(weight * Qs_arr, r_grid)
                    if mu_s_km1 > 0
                    else 0.0
                )

                print(
                    f"  WL: {wl} um -> mu_e: {mu_e_km1:.2f} km^-1, w0: {w0:.4f}, g: {g_avg:.4f}"
                )
                mc = PhotonTransportMC(
                    mu_e_km1,
                    w0,
                    g_avg,
                    z_max,
                    num_photons=num_photons,
                    analog_absorption=False,
                )
                _, _, b_ang, b_wgt = mc.run(return_weights=True)
                deg_b, int_b = PhotonTransportMC.calculate_intensity(
                    b_ang, b_wgt, num_photons, num_bins=60
                )

                dataset[f"{prefix}_wl_{wl}_backward_angle_deg"] = deg_b
                dataset[f"{prefix}_wl_{wl}_backward_intensity"] = int_b

    full_metadata = serialize_metadata_for_json(sweep_type, param_values, fixed_params)
    save_intensity_dataset(actual_data_path, dataset, full_metadata)

    return dataset


def plot_intensity_sweep(
    dataset,
    param_values,
    param_label,
    sweep_prefix,
    save=False,
    save_filename=None,
    fixed_params=None,
    sweep_type=None,
):
    colors = ["black", "red", "royalblue", "limegreen"]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Scattering Intensity vs Angle across Albedos", y=0.98)

    for val, color in zip(param_values, colors[: len(param_values)]):
        deg_f = dataset[f"{sweep_prefix}_{val}_forward_angle_deg"]
        int_f = dataset[f"{sweep_prefix}_{val}_forward_intensity"]
        deg_b = dataset[f"{sweep_prefix}_{val}_backward_angle_deg"]
        int_b = dataset[f"{sweep_prefix}_{val}_backward_intensity"]
        ax1.plot(deg_f, int_f, color=color, label=f"{param_label} = {val}", lw=2)
        ax2.plot(deg_b, int_b, color=color, label=f"{param_label} = {val}", lw=2)

    ax1.set_title(
        "Forward Scattering",
    )
    ax2.set_title(
        "Backward Scattering",
    )
    for ax in [ax1, ax2]:
        ax.set_xlabel("Scattering Angle (deg)")
        ax.set_ylabel("Normalized Intensity $I(\\theta)$")
        ax.set_xlim(0, 90)
        ax.grid(True)
        ax.legend()
    plt.tight_layout()
    if save and save_filename:
        actual_plot_path = get_metadata_plot_path(
            save_filename, sweep_type, param_values, fixed_params, PLOT_DIR
        )
        plt.savefig(actual_plot_path, dpi=300)
        print(f"Figure saved to {actual_plot_path}")
    plt.show()


def plot_backscatter_wavelength_sweep(
    dataset,
    wavelengths,
    fog_names,
    save=False,
    save_filename=None,
    fixed_params=None,
    sweep_type=None,
):
    colors = ["black", "red", "limegreen", "royalblue"]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Backscattering Intensity vs Angle across Wavelengths", y=0.98)
    fig.subplots_adjust(top=0.85, bottom=0.2, left=0.08, right=0.95, wspace=0.25)

    for ax, fog_name in zip(axes, fog_names):
        prefix = fog_name.lower().replace(" ", "_")
        for wl, color in zip(wavelengths, colors[: len(wavelengths)]):
            deg_b = dataset[f"{prefix}_wl_{wl}_backward_angle_deg"]
            int_b = dataset[f"{prefix}_wl_{wl}_backward_intensity"]
            ax.plot(deg_b, int_b, label=f"{wl} $\\mu$m", color=color, lw=2)
        ax.set_title(fog_name, fontsize=13, fontweight="bold")
        ax.set_xlabel("Backward Scattering Angle (deg)")
        ax.set_ylabel("Scattering Intensity $I(\\theta)$")
        ax.set_xlim(0, 90)
        ax.set_ylim(bottom=0)
        ax.grid()

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.01),
        ncol=len(wavelengths),
        frameon=False,
        fontsize=11,
    )
    if save and save_filename:
        actual_plot_path = get_metadata_plot_path(
            save_filename, sweep_type, wavelengths, fixed_params, PLOT_DIR
        )
        plt.savefig(actual_plot_path, dpi=300)
        print(f"Figure saved to {actual_plot_path}")
    plt.show()


### **7.2 Albedo Sweep: Reproduction of Figure 5**


Below we reproduce **Figure 5** of Xu et al. (2023), sweeping single-scattering albedo $\omega_0 \in \{0.1, 0.3, 0.6\}$ while keeping asymmetry factor fixed at $g=0.75$ through a $50\,\text{m}$ fog layer ($\mu_e = 39.12\,\text{km}^{-1}$). Higher albedo dramatically elevates both forward and backward scattering intensities due to reduced internal absorption.


In [ ]:
albedo_params = {
    "g": 0.75,
    "mu_e": 39.12,
    "z_max": 0.05,
    "num_photons": 500000,
    "metadata": {"description": "Figure 5 Albedo Sweep"},
}
albedo_data = run_or_load_intensity_sweep(
    "albedo", [0.1, 0.3, 0.6], albedo_params, DATA_DIR / "intensity_albedo_sweep.npz"
)
plot_intensity_sweep(
    albedo_data,
    [0.1, 0.3, 0.6],
    "$\\omega_0$",
    "albedo",
    save=True,
    save_filename="albedo_sweep.png",
    fixed_params=albedo_params,
    sweep_type="albedo",
)


### **7.3 Asymmetry-Factor Sweep: Reproduction of Figure 6**


Below we reproduce **Figure 6** of Xu et al. (2023), sweeping asymmetry factor $g \in \{0.25, 0.50, 0.75\}$ at fixed albedo $\omega_0=0.9$. As $g$ increases towards $1.0$, forward scattering intensity surges exponentially at narrow forward angles ($\theta \to 0^\circ$), while backscattering intensity drops corresponding to stronger forward persistence.


In [ ]:
g_params = {
    "albedo": 0.9,
    "mu_e": 39.12,
    "z_max": 0.05,
    "num_photons": 1000000,
    "metadata": {"description": "Figure 6 Asymmetry Factor Sweep"},
}
g_data = run_or_load_intensity_sweep(
    "g", [0.25, 0.50, 0.75], g_params, DATA_DIR / "intensity_g_sweep.npz"
)
plot_intensity_sweep(
    g_data,
    [0.25, 0.50, 0.75],
    "$g$",
    "g",
    save=True,
    save_filename="asymmetry_sweep.png",
    fixed_params=g_params,
    sweep_type="g",
)


### **7.4 Backscattering by Fog Type and Wavelength: Reproduction of Figure 7**


Below we reproduce **Figure 7**, evaluating backscattering intensity profiles across laser wavelengths ($0.86, 0.91, 1.06, 10.6\,\mu\text{m}$) in a $500\,\text{m}$ thick fog layer ($V = 0.6\,\text{km}$). 



In [ ]:
bs_params = {
    "visibility_km": 0.6,
    "z_max": 0.5,
    "num_photons": 500000,
    "fog_models": {"Radiation Fog": n_rad, "Advection Fog": n_adv},
    "r_grid": np.logspace(-3, 2.1, 1000),
    "ref_indices": FOG_REF_INDEX,
    "metadata": {"description": "Figure 7 Backscatter Wavelength Sweep"},
}
bs_data = run_or_load_intensity_sweep(
    "backscatter_wavelength",
    [0.86, 0.91, 1.06, 10.6],
    bs_params,
    DATA_DIR / "backscatter_wavelength_sweep.npz",
)
plot_backscatter_wavelength_sweep(
    bs_data,
    [0.86, 0.91, 1.06, 10.6],
    ["Radiation Fog", "Advection Fog"],
    save=True,
    save_filename="backscatter_wavelength_sweep.png",
    fixed_params=bs_params,
    sweep_type="backscatter_wavelength",
)


## 8. Discussion & Physical Interpretation of Simulation Results


Our numerical framework couples exact Lorenz-Mie single-scattering calculations with a 3D Photon Weight Tracking Monte Carlo algorithm. By systematically examining both microscopic droplet interactions and macroscopic photon transport, the figures and animations generated throughout this notebook reveal distinct physical mechanisms dictating laser beam propagation through advection and radiation fogs.

### **8.1 Microphysics and Bulk Optical Properties (Figures 1–4)**

#### **Fog Particle Size Distributions $n(r)$ (Figure 1)**
* **What the plot signifies:** The modified Gamma distribution curves illustrate a fundamental structural contrast: **Advection fog** possesses a broad distribution with significant droplet populations at larger radii ($r > 10\,\mu\text{m}$), whereas **Radiation fog** is heavily skewed toward dense concentrations of sub-micron and small microscopic droplets ($r < 4\,\mu\text{m}$).
* **Physical Implication:** The dimensionless size parameter $\alpha = 2\pi r / \lambda$ governs scattering physics. Because advection fog droplets are physically larger, their interaction with near-IR lasers ($0.86\text{ to }1.315\,\mu\text{m}$) lies firmly in the geometric optical limit ($\alpha \gg 1$). Conversely, radiation fog droplets approach the resonant Mie scattering regime ($\alpha \approx 1$) when probed by longer infrared wavelengths ($10.6\,\mu\text{m}$).

#### **Extinction Coefficient $\langle \mu_e \rangle$ vs. Visibility (Figure 2 Reproduction)**
* **What the plot signifies:** Extinction exhibits a strict log-log linear inverse relationship with meteorological visibility $V$. For near-IR wavelengths ($0.86, 0.91, 1.06, 1.315\,\mu\text{m}$), the extinction curves overlap almost perfectly across both fog types. However, at $\lambda = 10.6\,\mu\text{m}$, the extinction curve diverges, showing significantly lower attenuation at higher visibilities ($V > 1\,\text{km}$).
* **Physical Implication:**
  * **Geometric Scattering Limit:** For near-IR wavelengths, because droplet sizes far exceed the wavelength ($2\pi r \gg \lambda$), the extinction efficiency asymptotes to the geometric limit ($Q_e \approx 2$). Attenuation is purely a function of total cross-sectional area and is virtually wavelength-independent.
  * **Mie Resonance Shift:** For the $10.6\,\mu\text{m}$ far-IR laser, smaller droplets (especially in radiation fog) yield size parameters near $\alpha \approx 1$, where scattering efficiency $Q_s$ drops sharply. Consequently, far-IR radiation penetrates dispersed or thin fogs with markedly lower total extinction than near-IR lasers.

#### **Single Scattering Albedo $\langle \omega_0 \rangle$ vs. Visibility (Figure 3 Reproduction)**
* **What the plot signifies:** Across all near-IR wavelengths, single-scattering albedo remains virtually at unity ($\langle \omega_0 \rangle > 0.999$) across all visibilities. In stark contrast, at $10.6\,\mu\text{m}$, $\langle \omega_0 \rangle$ suffers a severe deficit ($\approx 0.3\text{ to }0.6$), dropping even steeper at higher visibilities in radiation fog.
* **Physical Implication:**
  * **Near-IR Elastic Scattering:** Liquid water is transparent in the near-IR window. Beam attenuation in near-IR is almost 100% governed by **conservative elastic scattering**—photons are redirected in space without losing energy.
  * **Far-IR Thermal Absorption:** Liquid water exhibits a strong vibrational absorption band near $10.6\,\mu\text{m}$ (imaginary refractive index $\kappa \approx 0.071$). When a $10.6\,\mu\text{m}$ photon interacts with a fog droplet, it is overwhelmingly absorbed rather than scattered. As visibility increases and droplet sizes shrink relative to $\lambda$, absorption dominates over scattering ($Q_a \gg Q_s$), causing the survival probability $\langle \omega_0 \rangle$ to plummet.

#### **Asymmetry Factor $\langle g \rangle$ vs. Visibility (Figure 4 Reproduction)**
* **What the plot signifies:** Near-IR wavelengths maintain consistently high asymmetry factors ($\langle g \rangle \approx 0.75\text{ to }0.88$), indicating strong forward directional bias. For $10.6\,\mu\text{m}$, $\langle g \rangle$ declines sharply toward zero as visibility increases.
* **Physical Implication:** Large droplets act as microscopic forward-focusing lenses for near-IR light, deflecting photons primarily at narrow forward angles ($\theta < 15^\circ$). When the laser wavelength ($10.6\,\mu\text{m}$) exceeds the average droplet dimensions of thin radiation fog, scattering transitions toward the isotropic Rayleigh regime ($\langle g \rangle \to 0$), scattering surviving photons uniformly across all directions.


### **8.2 Spatial Trajectory & Diffusion Dynamics (2D/3D Plots & Animations)**

#### **2D ($X-Z$) & 3D ($X-Y-Z$) Trajectory Visualizations and Rotating Animations**
* **What the visualizations/animations signify:** As collimated photon packets enter the fog medium at $(0,0,0)$ along the $Z$-axis, successive stochastic scattering events cause the beam to lose its initial spatial coherence, broadening laterally into a characteristic **"teardrop" or pear-shaped diffusion envelope**.
* **Physical Implication:**
  * **Ballistic to Diffusive Transition:** Near the entry boundary ($Z < 50\,\text{m}$), unscattered ballistic photons maintain high intensity and tight collimation along the optical axis. Deeper into the medium, multiple scattering dominates.
  * **Anisotropic Diffusion:** Because $\langle g \rangle > 0.75$, individual scattering events deflect photons only slightly from their forward trajectory. This allows the core of the beam to penetrate deeply before lateral diffusion ($X-Y$ spreading) disperses the optical power across a wider volume.
  * **FSO System Design:** For Free-Space Optical (FSO) communications, this teardrop expansion indicates that receiving apertures must be sufficiently wide to capture laterally diffused multi-path photons when operating beyond the ballistic limit ($\tau > 5$).

#### **Propagation Depth Evolution ($Z_{\max} = 100\,\text{m}, 250\,\text{m}, 500\,\text{m}$)**
* **What the plot signifies:** Side-by-side trajectory comparisons demonstrate that as the propagation distance increases, the lateral beam spread expands exponentially while the density of straight ballistic trajectories approaches zero.
* **Physical Implication:** Illustrates the optical depth ($\tau = \mu_e Z_{\max}$) limit. Once optical depth exceeds $\tau \approx 10$, spatial coherence is completely destroyed, and the laser beam transforms into an uncollimated, diffuse optical halo.


### **8.3 Angular Scattering Distributions & Parameter Sweeps (Figures 5–7)**

#### **Albedo Sensitivity Sweep $\omega_0 \in \{0.1, 0.3, 0.6\}$ (Figure 5 Reproduction)**
* **What the plot signifies:** At a fixed asymmetry factor ($g=0.75$), elevating the single-scattering albedo $\omega_0$ exponentially increases both forward ($I(\theta_f)$) and backward ($I(\theta_b)$) normalized scattering intensities across all angles. Forward intensity peaks sharply near $\theta_f \to 0^\circ$ and exceeds backscattering intensity by two to three orders of magnitude.
* **Physical Implication:** Albedo acts as a **multiplicative survival weight** during multiple scattering. In low-albedo media ($\omega_0 = 0.1$, corresponding to absorbing aerosols or $10.6\,\mu\text{m}$ in dense fog), photons undergoing multi-order scattering are rapidly extinguished by internal absorption before they can exit the medium. High albedo ($\ \omega_0 \to 1.0$) allows photons to survive dozens of scattering collisions, accumulating into intense forward halos and measurable diffuse backscatter.

#### **Asymmetry Factor Sweep $g \in \{0.25, 0.50, 0.75\}$ (Figure 6 Reproduction)**
* **What the plot signifies:** At a fixed albedo ($\omega_0=0.9$), increasing the asymmetry factor $g$ toward $1.0$ causes the narrow forward scattering intensity ($\theta_f \to 0^\circ$) to surge dramatically while simultaneously depressing the wide-angle and backward scattering profile ($\theta_b > 0^\circ$).
* **Physical Implication:** Demonstrates **angular photon budget redistribution**. Because the total surviving photon energy is fixed by $\omega_0$, a higher asymmetry factor acts as a directional funnel, channeling photons into a tight forward cone. Consequently, fewer photons are scattered at wide or backward angles. For LIDAR systems, higher $g$ values mean superior forward penetration but a significantly weakened backscattered return signal from the fog layer itself.

#### **Backscatter Wavelength Sensitivity Across Fog Types (Figure 7 Reproduction)**
* **What the plot signifies:** In a $500\,\text{m}$ fog layer ($V=0.6\,\text{km}$), **Advection fog generates substantially higher backscattering intensity at narrow angles ($\theta_b < 20^\circ$) than Radiation fog across all near-IR wavelengths** ($0.86, 0.91, 1.06\,\mu\text{m}$). Conversely, the $10.6\,\mu\text{m}$ far-IR backscatter signal is virtually flat and near zero for both fog types.
* **Physical Implication:**
  * **Backscatter Optical Glare:** Advection fog's larger mean droplet size ($r > 10\,\mu\text{m}$) produces strong backward diffraction lobes and glories under Mie theory. For near-IR LIDAR or optical receivers, advection fog creates intense **backscatter glare (optical noise)** near the transmitter axis, which can blind co-located sensors or cause false reflections.
  * **Far-IR Backscatter Immunity:** At $10.6\,\mu\text{m}$, because single-scattering albedo is low ($\omega_0 \approx 0.3$), photons penetrating the fog are absorbed before they can undergo the multiple backward collisions required to return to the source. This makes far-IR systems immune to fog-induced backscatter glare.


### **8.4 Engineering & System Architecture Implications (FSO vs. LIDAR)**

| Performance Metric | Near-IR Lasers ($0.86\text{ to }1.315\,\mu\text{m}$) | Far-IR Lasers ($10.6\,\mu\text{m}$) |
| :--- | :--- | :--- |
| **Primary Attenuation Mechanism** | Multiple Elastic Scattering ($\omega_0 \approx 1.0$) | Thermal Absorption ($\omega_0 \approx 0.3 - 0.6$) |
| **Penetration in Moderate/Dense Fog** | High geometric scattering loss ($\mu_e$ is high), but surviving photons maintain signal via forward diffusion ($g > 0.8$). | Lower Mie scattering extinction in thin fog ($\mu_e$ is lower), but rapid exponential drop-off over distance due to absorption ($Q_a \gg Q_s$). |
| **Backscatter Glare / Sensor Noise** | **Severe** (Especially in Advection fog due to large droplet diffraction peaks). | **Negligible** (Absorbed before backward reflection can occur). |
| **Optimal Application Domain** | Long-range FSO links with wide-aperture receivers to collect diffused forward halos. | Short-range precision LIDAR and secure tactical links requiring zero backscatter clutter. |


### **8.5 Numerical Methodology, Assumptions & Error Analysis**

1. **Henyey-Greenstein Phase Function vs. Exact Mie Glories:** While `miepython` provides exact single-scattering efficiencies ($Q_e, Q_s, g$), our Monte Carlo trajectory engine utilizes the Henyey-Greenstein (HG) phase function for angular deflection. HG matches the first angular moment ($\langle g \rangle$) exactly and excels at modeling macroscopic multiple-scattering diffusion. However, because HG is a smooth analytical approximation, it averages out fine oscillatory Mie phase features, such as primary/secondary rainbow angles ($\theta \approx 138^\circ$) and exact backscatter glory peaks ($\theta \to 180^\circ$).
2. **Monte Carlo Variance & Implicit Weight Tracking:** In Monte Carlo radiative transfer, statistical noise scales inversely with photon count ($\sigma \propto 1/\sqrt{N}$). Capturing deep backscattering into narrow angular bins ($\Delta \Omega_i$) is statistically rare. By implementing **implicit photon weight reduction** (analog absorption suppression) combined with mild **Savitzky-Golay polynomial filtering** (`savgol_filter`), our simulation successfully preserves true angular peak geometries while eliminating stochastic high-frequency jitter without requiring computationally prohibitive photon counts ($N > 10^7$).
3. **Plane-Parallel Homogeneity:** The model assumes the fog layer is infinitely extended horizontally and uniform along the propagation axis $Z$. In real-world atmospheric conditions, localized turbulence and vertical moisture gradients introduce spatial inhomogeneity, which can cause beam wandering and localized scintillation beyond the mean scattering behavior captured here.

# 9. References

1. **Xu, Q., et al. (2023)**. *The Multiple Scattering of Laser Beam Propagation in Advection Fog and Radiation Fog*. International Journal of Optics, 2023, 1-15. [https://doi.org/10.1155/2023/5594038](https://doi.org/10.1155/2023/5594038)
2. **Prahl, S. (2023)**. *miepython: Pure Python Lorenz-Mie Scattering for Homogeneous Spheres*. Read the Docs. [https://miepython.readthedocs.io/en/latest/](https://miepython.readthedocs.io/en/latest/)
3. **Henyey, L. G., & Greenstein, J. L. (1941)**. *Diffuse radiation in the galaxy*. Astrophysical Journal, 93, 70-83. [https://doi.org/10.1086/144246](https://doi.org/10.1086/144246)
4. **Kim, I. I., McArthur, B., & Korevaar, E. J. (2001)**. *Comparison of laser beam propagation at 785 nm and 1550 nm in fog and haze for optical wireless communications*. Proceedings of SPIE, 4214, 26-37. [https://doi.org/10.1117/12.417512](https://doi.org/10.1117/12.417512)
